# Week 4: O*NET and CIP Integration

## Objective

The objective of Week 4 is to integrate the selected Canadian occupations from Job Bank with O*NET occupational information and CIP educational pathways.

The analysis will:

1. Load and audit the O*NET datasets.
2. Build occupational profiles for the selected occupations.
3. Extract skills, knowledge, abilities, tasks, education, training, and Job Zone information.
4. Establish the NOC-to-O*NET occupational mapping.
5. Validate occupational mappings.
6. Integrate CIP education pathways.
7. Create an integrated occupation profile for the recommendation system.

This stage extends the Week 3 diagnostic labour-market analysis into occupational skills and educational pathways.

In [2]:
# Import required libraries

import pandas as pd
import numpy as np
import os
import glob

# Configure pandas display options for notebook analysis

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

In [3]:
# Define project paths

# Main project directory
PROJECT_ROOT = (
    r"C:\Users\Admin\Capstone_Project"
)

# Path containing the CIP dataset and other raw data files
RAW_DATA_PATH = (
    os.path.join(
        PROJECT_ROOT,
        "Data",
        "Raw_Data"
    )
)

# Path containing the 46 O*NET CSV files
ONET_PATH = (
    os.path.join(
        RAW_DATA_PATH,
        "db_onet_csv"
    )
)

# Project output directory
OUTPUT_PATH = (
    os.path.join(
        PROJECT_ROOT,
        "Outputs"
    )
)

# Create the output directory if it does not already exist
os.makedirs(
    OUTPUT_PATH,
    exist_ok=True
)

# Display paths for verification
print("Project Root:")
print(PROJECT_ROOT)

print("\nRaw Data Path:")
print(RAW_DATA_PATH)

print("\nO*NET Path:")
print(ONET_PATH)

print("\nOutput Path:")
print(OUTPUT_PATH)

Project Root:
C:\Users\Admin\Capstone_Project

Raw Data Path:
C:\Users\Admin\Capstone_Project\Data\Raw_Data

O*NET Path:
C:\Users\Admin\Capstone_Project\Data\Raw_Data\db_onet_csv

Output Path:
C:\Users\Admin\Capstone_Project\Outputs


In [4]:
# Audit O*NET dataset files

onet_files = sorted(os.listdir(ONET_PATH))

print("Number of files:", len(onet_files))
print()

for file in onet_files:
    print(file)

Number of files: 46

Read Me.txt
abilities.csv
abilities_to_work_activities.csv
abilities_to_work_context.csv
career_interest_type_keywords.csv
career_interest_types.csv
content_model_reference.csv
education.csv
education_categories.csv
emerging_tasks.csv
essential_skills.csv
essential_skills_to_work_activities.csv
essential_skills_to_work_context.csv
gwas_to_iwas.csv
gwas_to_iwas_to_dwas.csv
interests_illustrative_activities.csv
interests_illustrative_occupations.csv
job_titles.csv
job_zone_reference.csv
job_zones.csv
knowledge.csv
level_scale_anchors.csv
occupation_data.csv
occupation_level_metadata.csv
related_occupations.csv
sample_of_reported_titles.csv
scales_reference.csv
software_skills.csv
specific_interest_areas.csv
specific_interest_areas_to_career_interest_types.csv
survey_booklet_locations.csv
task_categories.csv
task_ratings.csv
task_statements.csv
tasks_to_dwas.csv
training_and_experience.csv
training_and_experience_categories.csv
transferable_skills.csv
transferable_ski

In [5]:
# Load the main O*NET datasets required for occupational analysis

occupation = pd.read_csv(
    os.path.join(ONET_PATH, "occupation_data.csv")
)

skills = pd.read_csv(
    os.path.join(ONET_PATH, "essential_skills.csv")
)

knowledge = pd.read_csv(
    os.path.join(ONET_PATH, "knowledge.csv")
)

abilities = pd.read_csv(
    os.path.join(ONET_PATH, "abilities.csv")
)

education = pd.read_csv(
    os.path.join(ONET_PATH, "education.csv")
)

job_zones = pd.read_csv(
    os.path.join(ONET_PATH, "job_zones.csv")
)

tasks = pd.read_csv(
    os.path.join(ONET_PATH, "task_statements.csv")
)

training = pd.read_csv(
    os.path.join(ONET_PATH, "training_and_experience.csv")
)

print("All main O*NET datasets loaded successfully.")

All main O*NET datasets loaded successfully.


In [6]:
# Audit the dimensions of the main O*NET datasets

# Store all loaded O*NET DataFrames in a single dictionary

datasets = {
    "Occupation": occupation,
    "Skills": skills,
    "Knowledge": knowledge,
    "Abilities": abilities,
    "Education": education,
    "Job Zones": job_zones,
    "Tasks": tasks,
    "Training and Experience": training
}

# Display the number of rows and columns for each dataset

for name, df in datasets.items():
    print("=" * 70)
    print(name)
    print("=" * 70)
    print("Rows:", df.shape[0])
    print("Columns:", df.shape[1])
    print()

Occupation
Rows: 1016
Columns: 3

Skills
Rows: 17880
Columns: 15

Knowledge
Rows: 59004
Columns: 15

Abilities
Rows: 92976
Columns: 15

Education
Rows: 11100
Columns: 15

Job Zones
Rows: 923
Columns: 5

Tasks
Rows: 18796
Columns: 8

Training and Experience
Rows: 26025
Columns: 15



In [7]:
# Audit the column names of each O*NET dataset

for name, df in datasets.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    
    print(df.columns.tolist())


Occupation
['O*NET-SOC Code', 'Title', 'Description']

Skills
['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']

Knowledge
['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']

Abilities
['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']

Education
['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Category', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Date', 'Domain Source']

Job Zones
['O*NET-SOC Code'

In [8]:
# Preview the O*NET occupation reference dataset

occupation.head()

,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide overall direction of companies or private and publi...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, shareholders, customers, and employees to address su..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of public or private sector organizations, overseeing..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes at the local, tribal, state, or federal level. In..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising policies and programs or produce collateral materials, s..."


In [9]:
# Inspect the structure and data types of the O*NET occupation dataset

occupation.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1016 entries, 0 to 1015
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   O*NET-SOC Code  1016 non-null   object
 1   Title           1016 non-null   object
 2   Description     1016 non-null   object
dtypes: object(3)
memory usage: 23.9+ KB


In [10]:
# Generate descriptive statistics for the O*NET occupation dataset

occupation.describe(include="all").T

,count,unique,top,freq
O*NET-SOC Code,1016,1016,11-1011.00,1
Title,1016,1016,Chief Executives,1
Description,1016,1016,Determine and formulate policies and provide overall direction of companies or private and publi...,1


In [11]:
# Audit missing values across the main O*NET datasets

for name, df in datasets.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)

# Count missing values in each column
    missing = df.isna().sum()
# Keep only columns containing missing values
    missing = missing[missing > 0].sort_values(ascending=False)

 # Display the 20 columns with the most missing values
    print(missing.head(20))


Occupation
Series([], dtype: int64)

Skills
Not Relevant    8940
dtype: int64

Knowledge
Not Relevant          29502
Lower CI Bound        15604
Upper CI Bound        15604
Standard Error        14916
Recommend Suppress    13926
N                      1056
dtype: int64

Abilities
Not Relevant    46488
dtype: int64

Education
Lower CI Bound        7477
Upper CI Bound        7477
Standard Error        2720
Recommend Suppress    2720
Category               564
dtype: int64

Job Zones
Series([], dtype: int64)

Tasks
Incumbents Responding    1190
Task Type                 845
dtype: int64

Training and Experience
Lower CI Bound        11912
Upper CI Bound        11912
Standard Error         6290
Recommend Suppress     6290
Category                563
dtype: int64


In [12]:
# Define the occupations selected during Weeks 2–3
# for O*NET and CIP integration

selected_occupations = [
    "Administrative Assistant",
    "Bookkeeper",
    "Continuing Care Assistant",
    "Delivery Driver",
    "Driver, Truck",
    "Food Service Supervisor",
    "Information Technology (IT) Analyst",
    "Inside Sales Representative",
    "Licensed Practical Nurse (L.P.N.)",
    "Office Administrator",
    "Office Manager",
    "Restaurant Manager",
    "Retail Sales Associate",
    "Secondary School Teacher",
    "Software Developer"
]

print("Number of selected occupations:", len(selected_occupations))

for i, occupation_name in enumerate(selected_occupations, start=1):
    print(f"{i}. {occupation_name}")

Number of selected occupations: 15
1. Administrative Assistant
2. Bookkeeper
3. Continuing Care Assistant
4. Delivery Driver
5. Driver, Truck
6. Food Service Supervisor
7. Information Technology (IT) Analyst
8. Inside Sales Representative
9. Licensed Practical Nurse (L.P.N.)
10. Office Administrator
11. Office Manager
12. Restaurant Manager
13. Retail Sales Associate
14. Secondary School Teacher
15. Software Developer


In [13]:
# Audit O*NET dataset dimensions and column structure

# Store the main O*NET datasets in a dictionary for systematic auditing

datasets = {
    "Occupation": occupation,
    "Skills": skills,
    "Knowledge": knowledge,
    "Abilities": abilities,
    "Education": education,
    "Job Zones": job_zones,
    "Tasks": tasks,
    "Training and Experience": training
}

# Display the shape and column names of each dataset

for name, df in datasets.items():
    print("=" * 70)
    print(name)
    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())
    print()

Occupation
Shape: (1016, 3)
Columns: ['O*NET-SOC Code', 'Title', 'Description']

Skills
Shape: (17880, 15)
Columns: ['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']

Knowledge
Shape: (59004, 15)
Columns: ['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']

Abilities
Shape: (92976, 15)
Columns: ['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']

Education
Shape: (11100, 15)
Columns: ['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Category', 'Data Value

In [14]:
# Inspect the O*NET occupation reference dataset

print("Shape:", occupation.shape)
print("\nColumns:")
print(occupation.columns.tolist())

print("\nFirst 10 rows:")
display(occupation.head(10))

Shape: (1016, 3)

Columns:
['O*NET-SOC Code', 'Title', 'Description']

First 10 rows:


,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide overall direction of companies or private and publi...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, shareholders, customers, and employees to address su..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of public or private sector organizations, overseeing..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes at the local, tribal, state, or federal level. In..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising policies and programs or produce collateral materials, s..."
5,11-2021.00,Marketing Managers,"Plan, direct, or coordinate marketing policies and programs, such as determining the demand for ..."
6,11-2022.00,Sales Managers,"Plan, direct, or coordinate the actual distribution or movement of a product or service to the c..."
7,11-2032.00,Public Relations Managers,"Plan, direct, or coordinate activities designed to create or maintain a favorable public image o..."
8,11-2033.00,Fundraising Managers,"Plan, direct, or coordinate activities to solicit and maintain funds for special projects or non..."
9,11-3012.00,Administrative Services Managers,"Plan, direct, or coordinate one or more administrative services of an organization, such as reco..."


In [15]:
# Generate possible O*NET occupation matches

# This step uses title similarity to identify candidate O*NET occupations for each selected Job Bank occupation.
# The matches must be manually validated before being used as the final occupational mapping.

import difflib

onet_titles = occupation["Title"].dropna().unique().tolist()

for selected in selected_occupations:
    
    matches = difflib.get_close_matches(
        selected,
        onet_titles,
        n=5,
        cutoff=0.3
    )
    
    print("=" * 70)
    print("Job Bank occupation:", selected)
    print("Possible O*NET matches:")
    
    for match in matches:
        print("  -", match)

Job Bank occupation: Administrative Assistant
Possible O*NET matches:
  - Administrative Services Managers
  - Legal Secretaries and Administrative Assistants
  - Medical Secretaries and Administrative Assistants
  - Statistical Assistants
  - Anesthesiologist Assistants
Job Bank occupation: Bookkeeper
Possible O*NET matches:
  - Bakers
  - Roofers
  - Coroners
  - Bicycle Repairers
  - Photographers
Job Bank occupation: Continuing Care Assistant
Possible O*NET matches:
  - Nursing Assistants
  - Dental Assistants
  - Statistical Assistants
  - Physician Assistants
  - Surgical Assistants
Job Bank occupation: Delivery Driver
Possible O*NET matches:
  - Taxi Drivers
  - Light Truck Drivers
  - Pile Driver Operators
  - Designers, All Other
  - Floral Designers
Job Bank occupation: Driver, Truck
Possible O*NET matches:
  - Bus Drivers, School
  - Driver/Sales Workers
  - Taxi Drivers
  - Order Clerks
  - Pile Driver Operators
Job Bank occupation: Food Service Supervisor
Possible O*NET ma

In [16]:
# Build a structured O*NET candidate mapping table

# Each selected Job Bank occupation is matched against possible
# O*NET occupation titles. The resulting candidates are stored with their rank, O*NET-SOC code, and description for validation.

candidate_mappings = []

# Generate candidate O*NET occupations for each selected occupation

for selected in selected_occupations:
    
    matches = difflib.get_close_matches(
        selected,
        onet_titles,
        n=5,
        cutoff=0.3
    )
 
# Store information for each candidate match  
    for rank, match in enumerate(matches, start=1):
        
        row = occupation[
            occupation["Title"] == match
        ].iloc[0]
        
        candidate_mappings.append({
            "job_bank_occupation": selected,
            "candidate_rank": rank,
            "onet_title": match,
            "onet_soc_code": row["O*NET-SOC Code"],
            "onet_description": row["Description"]
        })


# Convert candidate mappings into a DataFrame

onet_candidates = pd.DataFrame(candidate_mappings)

display(onet_candidates)

,job_bank_occupation,candidate_rank,onet_title,onet_soc_code,onet_description
0,Administrative Assistant,1,Administrative Services Managers,11-3012.00,"Plan, direct, or coordinate one or more administrative services of an organization, such as reco..."
1,Administrative Assistant,2,Legal Secretaries and Administrative Assistants,43-6012.00,"Perform secretarial duties using legal terminology, procedures, and documents. Prepare legal pap..."
2,Administrative Assistant,3,Medical Secretaries and Administrative Assistants,43-6013.00,"Perform secretarial duties using specific knowledge of medical terminology and hospital, clinic,..."
3,Administrative Assistant,4,Statistical Assistants,43-9111.00,Compile and compute data according to statistical formulas for use in statistical studies. May p...
4,Administrative Assistant,5,Anesthesiologist Assistants,29-1071.01,Assist anesthesiologists in the administration of anesthesia for surgical and non-surgical proce...
5,Bookkeeper,1,Bakers,51-3011.00,"Mix and bake ingredients to produce breads, rolls, cookies, cakes, pies, pastries, or other bake..."
6,Bookkeeper,2,Roofers,47-2181.00,"Cover roofs of structures with shingles, slate, asphalt, aluminum, wood, or related materials. M..."
7,Bookkeeper,3,Coroners,13-1041.06,"Direct activities such as autopsies, pathological and toxicological analyses, and inquests relat..."
8,Bookkeeper,4,Bicycle Repairers,49-3091.00,Repair and service bicycles.
9,Bookkeeper,5,Photographers,27-4021.00,"Photograph people, landscapes, merchandise, or other subjects. May use lighting equipment to enh..."


In [17]:
# Cell — Load the cleaned Job Bank data

import pandas as pd

# Define the path to the processed Job Bank dataset
JOB_BANK_PATH = (
    r"C:\Users\Admin\Capstone_Project"
    r"\Data\Processed"
    r"\job_bank_preliminary_clean.csv"
)

# Load the cleaned Job Bank dataset
df_jobbank = pd.read_csv(
    JOB_BANK_PATH,
    low_memory=False
)

# Display dataset dimensions
print(
    "Job Bank shape:",
    df_jobbank.shape
)

# Display available columns
print(
    "\nColumns:"
)

print(
    df_jobbank.columns.tolist()
)

Job Bank shape: (55091, 66)

Columns:
['posting_id', 'job_title', 'original_job_title', 'noc_2016_code', 'noc_2016_name', 'noc_2021_code', 'noc_2021_name', 'external_indicator', 'posting_date', 'vacancy_count', 'official_language', 'education_level', 'experience_level', 'government_type', 'placement_agency', 'naics', 'province', 'city', 'work_location_postal_code', 'economic__region', 'various_location', 'employment_type', 'employment_term', 'employment_term_start_date', 'employment_term_end_date', 'employment_term_oncall', 'employment_term_overtime', 'employment_term_day', 'employment_term_evening', 'employment_term_shift', 'employment_term_weekend', 'employment_term_night', 'employment_term_telework', 'employment_term_early', 'employment_term_flex', 'employment_term_morning', 'employment_term_tbd', 'salary_condition_detail', 'salary_period', 'salary_minimum', 'salary_maximum', 'salary_condition_collective', 'salary_condition_bonus', 'salary_condition_disability', 'salary_condition_gr

In [18]:
# Cell — Identify selected occupations in the Job Bank dataset

# Normalize occupation titles for consistent matching
selected_jobbank = df_jobbank[
    df_jobbank["job_title"]
    .str.strip()
    .str.lower()
    .isin(
        [
            occupation_name.lower()
            for occupation_name in selected_occupations
        ]
    )
].copy()

# Display the number of selected Job Bank records
print(
    "Selected occupation records:",
    selected_jobbank.shape
)

# Display the NOC classifications associated with
# the selected occupations
display(
    selected_jobbank[
        [
            "job_title",
            "noc_2021_code",
            "noc_2021_name",
            "noc_2016_code",
            "noc_2016_name"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        "job_title"
    )
)

Selected occupation records: (5629, 66)


,job_title,noc_2021_code,noc_2021_name,noc_2016_code,noc_2016_name
16515,Administrative Assistant,13110.0,Administrative assistants,1241.0,Administrative assistants
11792,Food service supervisor,62020.0,Food service supervisors,6311.0,Food service supervisors
22012,Retail sales associate,64100.0,Retail salespersons and visual merchandisers,6421.0,Retail salespersons
109,administrative assistant,13110.0,Administrative assistants,1241.0,Administrative assistants
379,bookkeeper,12200.0,Accounting technicians and bookkeepers,1311.0,Accounting technicians and bookkeepers
362,continuing care assistant,33102.0,"Nurse aides, orderlies and patient service associates",3413.0,"Nurse aides, orderlies and patient service associates"
11,delivery driver,75201.0,Delivery service drivers and door-to-door distributors,7514.0,Delivery and courier service drivers
136,"driver, truck",73300.0,Transport truck drivers,7511.0,Transport truck drivers
95,food service supervisor,62020.0,Food service supervisors,6311.0,Food service supervisors
584,information technology (IT) analyst,21222.0,Information systems specialists,2171.0,Information systems analysts and consultants


In [19]:
# Cell — Inspect actual Job Bank occupation names and NOC 2021 classifications

occupation_summary = (
    df_jobbank[
        [
            "job_title",
            "noc_2021_code",
            "noc_2021_name"
        ]
    ]
    .drop_duplicates()
    .sort_values(
        [
            "noc_2021_name",
            "job_title"
        ]
    )
)

# Display the number of unique occupation/title combinations
print(
    "Number of unique occupation/title combinations:",
    len(occupation_summary)
)

# Display the first 100 combinations for inspection
display(
    occupation_summary.head(100)
)

Number of unique occupation/title combinations: 6788


,job_title,noc_2021_code,noc_2021_name
19980,accommodation services manager,60031.0,Accommodation service managers
6218,accommodations manager,60031.0,Accommodation service managers
4426,assistant manager - accommodation services,60031.0,Accommodation service managers
45,"assistant manager, hotel",60031.0,Accommodation service managers
4750,camp manager,60031.0,Accommodation service managers
27304,campground manager,60031.0,Accommodation service managers
31260,"director, hotel",60031.0,Accommodation service managers
35588,front desk hotel manager,60031.0,Accommodation service managers
5989,front desk manager,60031.0,Accommodation service managers
24317,front desk manager - accommodation services,60031.0,Accommodation service managers


In [20]:
# Cell — Find actual Job Bank titles related to the selected occupations

# Define search terms representing the selected occupations
search_terms = [
    "software",
    "information technology",
    "IT analyst",
    "administrative assistant",
    "bookkeeper",
    "office administrator",
    "office manager",
    "restaurant manager",
    "food service",
    "retail sales",
    "inside sales",
    "truck driver",
    "delivery driver",
    "secondary school",
    "continuing care",
    "licensed practical nurse"
]


# Search both Job Bank titles and NOC 2021 occupation names
for term in search_terms:

    matches = occupation_summary[
        occupation_summary["job_title"]
        .str.contains(
            term,
            case=False,
            na=False
        )
        |
        occupation_summary["noc_2021_name"]
        .str.contains(
            term,
            case=False,
            na=False
        )
    ]

    print(
        "\n" + "=" * 70
    )

    print(
        "SEARCH:",
        term
    )

    print(
        "=" * 70
    )

    display(
        matches.head(20)
    )


SEARCH: software


,job_title,noc_2021_code,noc_2021_name
50009,computer software design manager,20012.0,Computer and information systems managers
12372,software development manager,20012.0,Computer and information systems managers
31014,software engineering manager,20012.0,Computer and information systems managers
45099,computer engineering project manager,21311.0,Computer engineers (except software engineers and designers)
26262,"engineer, hardware",21311.0,Computer engineers (except software engineers and designers)
486,hardware engineer,21311.0,Computer engineers (except software engineers and designers)
21808,infrastructure architect - information technology (IT),21311.0,Computer engineers (except software engineers and designers)
16519,network architect - computer systems,21311.0,Computer engineers (except software engineers and designers)
50564,network systems engineer,21311.0,Computer engineers (except software engineers and designers)
10216,prompt engineer,21311.0,Computer engineers (except software engineers and designers)



SEARCH: information technology


,job_title,noc_2021_code,noc_2021_name
4832,IT (information technology) business analyst,21221.0,Business systems specialists
13717,information technology (IT) business analyst,21221.0,Business systems specialists
49114,IT (information technology) development manager,20012.0,Computer and information systems managers
24418,IT (information technology) integration manager,20012.0,Computer and information systems managers
38133,information technology (IT) director,20012.0,Computer and information systems managers
8786,information technology (IT) implementation manager,20012.0,Computer and information systems managers
36139,information technology (it) system administrator,20012.0,Computer and information systems managers
3477,"manager, IT (information technology) implementation",20012.0,Computer and information systems managers
22534,technical program manager - information technology (IT),20012.0,Computer and information systems managers
21808,infrastructure architect - information technology (IT),21311.0,Computer engineers (except software engineers and designers)



SEARCH: IT analyst


...

,job_title,noc_2021_code,noc_2021_name
8624,"aide, teacher's",43100.0,Elementary and secondary school teacher assistants
18,"assistant, educational",43100.0,Elementary and secondary school teacher assistants
4238,"assistant, special education",43100.0,Elementary and secondary school teacher assistants
24823,"attendant, child care - elementary school",43100.0,Elementary and secondary school teacher assistants
968,educational assistant,43100.0,Elementary and secondary school teacher assistants
52559,elementary school teacher's aide,43100.0,Elementary and secondary school teacher assistants
35730,elementary school teacher's assistant,43100.0,Elementary and secondary school teacher assistants
160,homework assistant,43100.0,Elementary and secondary school teacher assistants
712,program assistant - education,43100.0,Elementary and secondary school teacher assistants
2381,"program assistant, education",43100.0,Elementary and secondary school teacher assistants



SEARCH: continuing care


,job_title,noc_2021_code,noc_2021_name
362,continuing care assistant,33102.0,"Nurse aides, orderlies and patient service associates"



SEARCH: licensed practical nurse


,job_title,noc_2021_code,noc_2021_name
3771,CNA (certified nursing assistant),32101.0,Licensed practical nurses
375,L.P.N. (licensed practical nurse),32101.0,Licensed practical nurses
49490,Nurse educator - cannabis,32101.0,Licensed practical nurses
5904,R.P.N. (registered practical nurse),32101.0,Licensed practical nurses
17443,graduate nursing assistant,32101.0,Licensed practical nurses
360,licensed practical nurse (L.P.N.),32101.0,Licensed practical nurses
3777,nursing assistant (registered - Québec),32101.0,Licensed practical nurses
54579,registered nursing assistant (R.N.A.),32101.0,Licensed practical nurses
265,registered practical nurse (R.P.N.),32101.0,Licensed practical nurses


In [21]:
# Cell — Create the selected occupation-to-NOC 2021 mapping

# This mapping links each selected Job Bank occupation to its corresponding NOC 2021 code for the Week 4 O*NET integration.

selected_noc21 = {
    "Software Developer": "21232",
    "Information Technology (IT) Analyst": "21222",
    "Administrative Assistant": "13110",
    "Bookkeeper": "12200",
    "Office Administrator": "13100",
    "Office Manager": "13100",
    "Restaurant Manager": "60030",
    "Food Service Supervisor": "62020",
    "Retail Sales Associate": "64100",
    "Inside Sales Representative": "64100",
    "Driver, Truck": "73300",
    "Delivery Driver": "75201",
    "Secondary School Teacher": "41220",
    "Continuing Care Assistant": "33102",
    "Licensed Practical Nurse (L.P.N.)": "32101"
}

# Convert the occupation-to-NOC mapping into a DataFrame

selected_noc = pd.DataFrame(
    list(selected_noc21.items()),
    columns=["selected_occupation", "noc_2021_code"]
)

selected_noc

,selected_occupation,noc_2021_code
0,Software Developer,21232
1,Information Technology (IT) Analyst,21222
2,Administrative Assistant,13110
3,Bookkeeper,12200
4,Office Administrator,13100
5,Office Manager,13100
6,Restaurant Manager,60030
7,Food Service Supervisor,62020
8,Retail Sales Associate,64100
9,Inside Sales Representative,64100


In [22]:
# Cell — Find NOC, SOC, and CIP crosswalk files

# Search the raw-data directory for classification and crosswalk
# files that may support the NOC-to-O*NET/SOC and CIP integration.

RAW_DATA_PATH = (
    r"C:\Users\Admin\Capstone_Project"
    r"\Data\Raw_Data"
)

# Store paths of potential classification/crosswalk files

crosswalk_files = []

# Recursively search the raw-data directory

for root, dirs, files in os.walk(RAW_DATA_PATH):
    for file in files:
        filename = file.lower()

# Identify files that may contain NOC, SOC, CIP,
# or occupational classification crosswalk information
        if (
            "crosswalk" in filename
            or "noc" in filename
            or "soc" in filename
            or "cip" in filename
        ):
            crosswalk_files.append(os.path.join(root, file))

# Display potential classification/crosswalk files

print("Potential classification/crosswalk files:")
print()

for file in sorted(crosswalk_files):
    print(file)

Potential classification/crosswalk files:

C:\Users\Admin\Capstone_Project\Data\Raw_Data\CIPCode2020.csv


In [23]:
# Cell — List all files in the Crosswalks folder

# This cell audits the available classification and crosswalk
# files before selecting the appropriate NOC, SOC, or CIP resource.

import os

# Define the Crosswalks folder path

CROSSWALK_PATH = (
    r"C:\Users\Admin\Capstone_Project"
    r"\Data\Crosswalks"
)


# List all files in the Crosswalks folder

all_crosswalk_files = sorted(os.listdir(CROSSWALK_PATH))

print("Number of files:", len(all_crosswalk_files))
print()

for file in all_crosswalk_files:
    print(file)

Number of files: 4

CIP_to_SOC
NOC2016_to_SOC2018
NOC2021_to_NOC2016
SOC2018_to_ONET


In [24]:
# Cell — Inspect the four classification crosswalk folders

# These folders provide the classification bridges needed to connect
# NOC 2021 occupations with O*NET and CIP educational pathways.

import os

crosswalk_folders = [
    "NOC2021_to_NOC2016",
    "NOC2016_to_SOC2018",
    "SOC2018_to_ONET",
    "CIP_to_SOC"
]

for folder in crosswalk_folders:
    folder_path = os.path.join(CROSSWALK_PATH, folder)
    
    print("=" * 70)
    print(folder)
    print("=" * 70)
    
    if os.path.exists(folder_path):
        files = sorted(os.listdir(folder_path))
        
        print("Number of files:", len(files))
        for file in files:
            print("  ", file)
    else:
        print("FOLDER NOT FOUND:", folder_path)
    
    print()

NOC2021_to_NOC2016
Number of files: 1
   noc2016v1_3-noc2021v1_0-eng.csv

NOC2016_to_SOC2018
Number of files: 1
   noc2016v1_3-soc2018us-eng.csv

SOC2018_to_ONET
Number of files: 1
   2019_to_SOC_Crosswalk.xlsx

CIP_to_SOC
Number of files: 1
   CIP2020_SOC2018_Crosswalk.xlsx



In [25]:
# Cell — Define classification crosswalk file paths

# These files provide the classification bridges required for
# NOC 2021 → NOC 2016 → SOC 2018 → O*NET integration,
# plus the CIP 2020 → SOC 2018 education pathway.

# Define the root Crosswalks directory
CROSSWALK_PATH = (
    r"C:\Users\Admin\Capstone_Project"
    r"\Data\Crosswalks"
)

NOC21_NOC16_FILE = os.path.join(
    CROSSWALK_PATH,
    "NOC2021_to_NOC2016",
    "noc2016v1_3-noc2021v1_0-eng.csv"
)

NOC16_SOC_FILE = os.path.join(
    CROSSWALK_PATH,
    "NOC2016_to_SOC2018",
    "noc2016v1_3-soc2018us-eng.csv"
)

SOC_ONET_FILE = os.path.join(
    CROSSWALK_PATH,
    "SOC2018_to_ONET",
    "2019_to_SOC_Crosswalk.xlsx"
)

CIP_SOC_FILE = os.path.join(
    CROSSWALK_PATH,
    "CIP_to_SOC",
    "CIP2020_SOC2018_Crosswalk.xlsx"
)

print("Crosswalk paths set.")

Crosswalk paths set.


In [26]:
# Cell — Load classification crosswalk datasets
# Load the four crosswalk resources required to establish the
# NOC 2021 → NOC 2016 → SOC 2018 → O*NET pathway and the
# CIP 2020 → SOC 2018 education pathway.

noc21_noc16 = pd.read_csv(
    NOC21_NOC16_FILE,
    encoding="latin1"
)

noc16_soc = pd.read_csv(
    NOC16_SOC_FILE,
    encoding="latin1"
)

soc_onet = pd.read_excel(SOC_ONET_FILE)

cip_soc = pd.read_excel(CIP_SOC_FILE)

print("All four crosswalks loaded successfully.")

All four crosswalks loaded successfully.


In [27]:
# Cell — Crosswalk data inventory

# Summarize the size of each classification crosswalk before
# inspecting column names and building the occupational mapping.

# Store the four crosswalk datasets in a dictionary

crosswalks = {
    "NOC 2021 → NOC 2016": noc21_noc16,
    "NOC 2016 → SOC 2018": noc16_soc,
    "SOC 2018 → O*NET": soc_onet,
    "CIP 2020 → SOC 2018": cip_soc
}

for name, data in crosswalks.items():
    print("=" * 70)
    print(name)
    print("Rows:", data.shape[0])
    print("Columns:", data.shape[1])

NOC 2021 → NOC 2016
Rows: 585
Columns: 7
NOC 2016 → SOC 2018
Rows: 1250
Columns: 6
SOC 2018 → O*NET
Rows: 1019
Columns: 4
CIP 2020 → SOC 2018
Rows: 7
Columns: 2


In [28]:
# Cell — Inspect crosswalk column names

# Review the exact field names in each crosswalk before performing
# any merges or NOC-to-SOC-to-O*NET mapping operations.

for name, data in crosswalks.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    print(data.columns.tolist())


NOC 2021 → NOC 2016
['ï»¿NOC 2016 V1.3 Code', 'NOC 2016 V1.3 Title', ' GSIM Type of Change', 'NOC 2021 V1.0 Code', 'NOC 2021 V1.0 Title', 'Notes', 'Unnamed: 6']

NOC 2016 → SOC 2018
['NOC 2016  Version 1.3 Code', 'NOC 2016  Version 1.3 Title', 'Partial', 'SOC 2018 (US) Code', 'SOC 2018 (US) Title', 'Explanatory Notes']

SOC 2018 → O*NET
['O*NET-SOC 2019 Occupation Listings', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3']

CIP 2020 → SOC 2018
['File Name', 'Description']


In [29]:
# Cell — Clean NOC 2021 → NOC 2016 crosswalk

# Remove possible UTF-8 BOM characters and leading/trailing whitespace
# from the crosswalk column names so they can be referenced reliably.

noc21_noc16.columns = (
    noc21_noc16.columns
    .str.replace("ï»¿", "", regex=False)
    .str.strip()
)

# Display the first few rows after cleaning
noc21_noc16.head()

,NOC 2016 V1.3 Code,NOC 2016 V1.3 Title,GSIM Type of Change,NOC 2021 V1.0 Code,NOC 2021 V1.0 Title,Notes,Unnamed: 6
0,11,Legislators,"VC1 - Code Change, VC2 - Name Change",10,Legislators,NaN,NaN
1,12,Senior government managers and officials,"VC1 - Code Change, VC2 - Name Change",11,Senior government managers and officials,NaN,NaN
2,13,"Senior managers - financial, communications and other business services","VC1 - Code Change, VC2 - Name Change",12,"Senior managers - financial, communications and other business services",NaN,NaN
3,14,"Senior managers - health, education, social and community services and membership organizations","VC1 - Code Change, VC2 - Name Change",13,"Senior managers - health, education, social and community services and membership organizations",NaN,NaN
4,15,"Senior managers - trade, broadcasting and other services, n.e.c.","VC1 - Code Change, VC2 - Name Change",14,"Senior managers - trade, broadcasting and other services",NaN,NaN


In [30]:
# Cell — Inspect NOC 2021 to NOC 2016 classification changes

# Display the key classification fields and the GSIM change type to understand how each NOC 2021 occupation maps to NOC 2016.

display(
    noc21_noc16[
        [
            "NOC 2021 V1.0 Code",
            "NOC 2021 V1.0 Title",
            "NOC 2016 V1.3 Code",
            "NOC 2016 V1.3 Title",
            "GSIM Type of Change"
        ]
    ].head(20)
)

,NOC 2021 V1.0 Code,NOC 2021 V1.0 Title,NOC 2016 V1.3 Code,NOC 2016 V1.3 Title,GSIM Type of Change
0,10,Legislators,11,Legislators,"VC1 - Code Change, VC2 - Name Change"
1,11,Senior government managers and officials,12,Senior government managers and officials,"VC1 - Code Change, VC2 - Name Change"
2,12,"Senior managers - financial, communications and other business services",13,"Senior managers - financial, communications and other business services","VC1 - Code Change, VC2 - Name Change"
3,13,"Senior managers - health, education, social and community services and membership organizations",14,"Senior managers - health, education, social and community services and membership organizations","VC1 - Code Change, VC2 - Name Change"
4,14,"Senior managers - trade, broadcasting and other services",15,"Senior managers - trade, broadcasting and other services, n.e.c.","VC1 - Code Change, VC2 - Name Change"
5,15,"Senior managers - construction, transportation, production and utilities",16,"Senior managers - construction, transportation, production and utilities","VC1 - Code Change, VC2 - Name Change"
6,10010,Financial managers,111,Financial managers,"VC1 - Code Change, VC2 - Name Change"
7,10011,Human resources managers,112,Human resources managers,"RC4.2 - Split off, RC5 - Transfer, VC1 - Code Change, VC2 - Name Change"
8,11200,Human resources professionals,112,Human resources managers,"RC5 - Transfer, VC1 - Code Change, VC2 - Name Change"
9,13110,Administrative assistants,112,Human resources managers,"RC5 - Transfer, VC1 - Code Change, VC2 - Name Change"


In [31]:
# Cell — Inspect workbook sheet names

# Review the available worksheets in the SOC → O*NET and
# CIP → SOC Excel workbooks before selecting the appropriate sheets.

# Load the SOC → O*NET workbook structure

soc_onet_excel = pd.ExcelFile(SOC_ONET_FILE)
cip_soc_excel = pd.ExcelFile(CIP_SOC_FILE)

print("SOC → O*NET sheets:")
print(soc_onet_excel.sheet_names)

print("\nCIP → SOC sheets:")
print(cip_soc_excel.sheet_names)

SOC → O*NET sheets:
['O-NET-SOC 2019 Occupation Listi']

CIP → SOC sheets:
['File Guide', 'CIP-SOC', 'SOC-CIP', 'New CIP', 'New SOC', 'Added Matches', 'Unmatched CIP Codes', 'Unmatched SOC Codes']


In [32]:
# Cell — Inspect workbook contents

# Review every worksheet in the SOC → O*NET and CIP → SOC
# workbooks, including sheet dimensions and sample rows.
# Header=None is used temporarily so that the original workbook
# structure can be inspected without assuming the header location.


# ================================================================
# Inspect SOC → O*NET workbook
# ================================================================

print("=" * 80)
print("SOC → O*NET")
print("=" * 80)

for sheet in soc_onet_excel.sheet_names:
    temp = pd.read_excel(SOC_ONET_FILE, sheet_name=sheet, header=None)
    
    print("\nSHEET:", sheet)
    print("Shape:", temp.shape)
    display(temp.head(10))

# ================================================================
# Inspect CIP → SOC workbook
# ================================================================

print("\n" + "=" * 80)
print("CIP → SOC")
print("=" * 80)

for sheet in cip_soc_excel.sheet_names:
    temp = pd.read_excel(CIP_SOC_FILE, sheet_name=sheet, header=None)
    
    print("\nSHEET:", sheet)
    print("Shape:", temp.shape)
    display(temp.head(10))

SOC → O*NET

SHEET: O-NET-SOC 2019 Occupation Listi
Shape: (1020, 4)


,0,1,2,3
0,O*NET-SOC 2019 Occupation Listings,NaN,NaN,NaN
1,Crosswalk O*NET-SOC 2019 to 2018 SOC,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,O*NET-SOC 2019 Code,O*NET-SOC 2019 Title,2018 SOC Code,2018 SOC Title
4,11-1011.00,Chief Executives,11-1011,Chief Executives
5,11-1011.03,Chief Sustainability Officers,11-1011,Chief Executives
6,11-1021.00,General and Operations Managers,11-1021,General and Operations Managers
7,11-1031.00,Legislators,11-1031,Legislators
8,11-2011.00,Advertising and Promotions Managers,11-2011,Advertising and Promotions Managers
9,11-2021.00,Marketing Managers,11-2021,Marketing Managers



CIP → SOC

SHEET: File Guide
Shape: (8, 2)


,0,1
0,File Name,Description
1,CIP-SOC,This file crosswalks 2020 CIP Codes to 2018 SOC Codes in ascending order by CIP Code.
2,SOC-CIP,This file crosswalks 2018 SOC Codes to 2020 CIP Codes in ascending order by SOC Code.
3,New CIP,This file contains only NEW 2020 CIP Codes. It crosswalks NEW 2020 CIP Codes to 2018 SOC Codes i...
4,New SOC,This file contains only NEW 2018 SOC Codes. It crosswalks NEW 2018 SOC Codes to 2020 CIP Codes i...
5,Added Matches,This file contains only NEW matches that were added after reviewing the 2010 CIP SOC Crosswalk.
6,Unmatched CIP Codes,This file contains CIP Codes that do not have a corresponding SOC Code.
7,Unmatched SOC Codes,This file contains SOC Codes that do not have a corresponding CIP Codes.



SHEET: CIP-SOC
Shape: (6098, 4)


...

,0,1,2,3
0,CIP2020Code,CIP2020Title,SOC2018Code,SOC2018Title
1,01.0101,"Agricultural Business and Management, General.",45-1011,"First-Line Supervisors of Farming, Fishing, and Forestry Workers"
2,01.0103,Agricultural Economics.,25-1063,"Economics Teachers, Postsecondary"
3,01.0105,Agricultural/Farm Supplies Retailing and Wholesaling.,41-4012,"Sales Representatives, Wholesale and Manufacturing, Except Technical and Scientific Products"
4,01.0306,Dairy Husbandry and Production.,25-9021,Farm and Home Management Educators
5,01.0307,Horse Husbandry/Equine Science and Management.,25-1041,"Agricultural Sciences Teachers, Postsecondary"
6,01.0307,Horse Husbandry/Equine Science and Management.,25-9021,Farm and Home Management Educators
7,01.0504,Dog/Pet/Animal Grooming.,25-1194,"Career/Technical Education Teachers, Postsecondary"
8,01.0601,"Applied Horticulture/Horticulture Operations, General.",25-9021,Farm and Home Management Educators
9,01.0601,"Applied Horticulture/Horticulture Operations, General.",37-3011,Landscaping and Groundskeeping Workers



SHEET: Unmatched CIP Codes
Shape: (195, 4)


,0,1,2,3
0,CIP2020Code,CIP2020Title,SOC2018Code,SOC2018Title
1,01.0508,Taxidermy/Taxidermist.,99-9999,NO MATCH
2,01.0599,"Agricultural and Domestic Animal Services, Other.",99-9999,NO MATCH
3,01.0699,"Applied Horticulture/Horticultural Business Services, Other.",99-9999,NO MATCH
4,01.0899,"Agricultural Public Services, Other.",99-9999,NO MATCH
5,01.1302,Pre-Veterinary Studies.,99-9999,NO MATCH
6,01.8299,"Veterinary Administrative Services, Other.",99-9999,NO MATCH
7,01.9999,"Agricultural/Animal/Plant/Veterinary Science and Related Fields, Other.",99-9999,NO MATCH
8,03.0199,"Natural Resources Conservation and Research, Other.",99-9999,NO MATCH
9,03.0210,Bioenergy.,99-9999,NO MATCH



SHEET: Unmatched SOC Codes
Shape: (181, 4)


,0,1,2,3
0,SOC2018Code,SOC2018Title,CIP2020Code,CIP2020Title
1,13-1074,Farm Labor Contractors,99.9999,NO MATCH
2,25-3031,"Substitute Teachers, Short-Term",99.9999,NO MATCH
3,25-3041,Tutors,99.9999,NO MATCH
4,27-1023,Floral Designers,99.9999,NO MATCH
5,27-2023,"Umpires, Referees, and Other Sports Officials",99.9999,NO MATCH
6,27-4099,"Media and Communication Equipment Workers, All Other",99.9999,NO MATCH
7,31-1132,Orderlies,99.9999,NO MATCH
8,31-9095,Pharmacy Aides,99.9999,NO MATCH
9,33-3041,Parking Enforcement Workers,99.9999,NO MATCH


In [33]:
# Cell — Load and clean the SOC 2018 → O*NET crosswalk

# Load the worksheet containing the SOC 2018 to O*NET occupation mapping.
# The workbook contains three rows of introductory information before the actual column headers.

soc_onet = pd.read_excel(
    SOC_ONET_FILE,
    sheet_name="O-NET-SOC 2019 Occupation Listi",
    skiprows=3
)

# Clean column names
soc_onet.columns = soc_onet.columns.str.strip()

print("Shape:", soc_onet.shape)
print()
print(soc_onet.columns.tolist())

display(soc_onet.head())

Shape: (1016, 4)

['O*NET-SOC 2019 Code', 'O*NET-SOC 2019 Title', '2018 SOC Code', '2018 SOC Title']


,O*NET-SOC 2019 Code,O*NET-SOC 2019 Title,2018 SOC Code,2018 SOC Title
0,11-1011.00,Chief Executives,11-1011,Chief Executives
1,11-1011.03,Chief Sustainability Officers,11-1011,Chief Executives
2,11-1021.00,General and Operations Managers,11-1021,General and Operations Managers
3,11-1031.00,Legislators,11-1031,Legislators
4,11-2011.00,Advertising and Promotions Managers,11-2011,Advertising and Promotions Managers


In [34]:
# Cell — Load CIP 2020 → SOC 2018 crosswalk

cip_soc = pd.read_excel(
    CIP_SOC_FILE,
    sheet_name="CIP-SOC",
    header=0
)

cip_soc.columns = cip_soc.columns.str.strip()

print("Shape:", cip_soc.shape)
print()
print(cip_soc.columns.tolist())

display(cip_soc.head())

Shape: (6097, 4)

['CIP2020Code', 'CIP2020Title', 'SOC2018Code', 'SOC2018Title']


,CIP2020Code,CIP2020Title,SOC2018Code,SOC2018Title
0,1.0,"Agriculture, General.",19-1011,Animal Scientists
1,1.0,"Agriculture, General.",19-1012,Food Scientists and Technologists
2,1.0,"Agriculture, General.",19-1013,Soil and Plant Scientists
3,1.0,"Agriculture, General.",19-4012,Agricultural Technicians
4,1.0,"Agriculture, General.",25-1041,"Agricultural Sciences Teachers, Postsecondary"


In [35]:
# Cell — Load and clean the CIP 2020 → SOC 2018 crosswalk

# Load the CIP-SOC worksheet that connects educational programs (CIP 2020) with related occupations (SOC 2018).
# This crosswalk will support the educational pathway component of the Week 4 integrated occupation profile.

soc_onet["2018 SOC Code"] = (
    soc_onet["2018 SOC Code"]
    .astype(str)
    .str.strip()
)

cip_soc["SOC2018Code"] = (
    cip_soc["SOC2018Code"]
    .astype(str)
    .str.strip()
)

print("SOC → O*NET examples:")
display(soc_onet.head())

print("\nCIP → SOC examples:")
display(cip_soc.head())

SOC → O*NET examples:


,O*NET-SOC 2019 Code,O*NET-SOC 2019 Title,2018 SOC Code,2018 SOC Title
0,11-1011.00,Chief Executives,11-1011,Chief Executives
1,11-1011.03,Chief Sustainability Officers,11-1011,Chief Executives
2,11-1021.00,General and Operations Managers,11-1021,General and Operations Managers
3,11-1031.00,Legislators,11-1031,Legislators
4,11-2011.00,Advertising and Promotions Managers,11-2011,Advertising and Promotions Managers



CIP → SOC examples:


,CIP2020Code,CIP2020Title,SOC2018Code,SOC2018Title
0,1.0,"Agriculture, General.",19-1011,Animal Scientists
1,1.0,"Agriculture, General.",19-1012,Food Scientists and Technologists
2,1.0,"Agriculture, General.",19-1013,Soil and Plant Scientists
3,1.0,"Agriculture, General.",19-4012,Agricultural Technicians
4,1.0,"Agriculture, General.",25-1041,"Agricultural Sciences Teachers, Postsecondary"


In [36]:
# Identify all pandas DataFrames currently stored in memory

df_names = [
    name for name in list(globals().keys())
    if isinstance(globals()[name], pd.DataFrame)
]

print("DataFrames currently in memory:")
for name in df_names:
    print(name)

DataFrames currently in memory:
_
__
___
occupation
skills
knowledge
abilities
education
job_zones
tasks
training
df
_8
_10
matches
onet_candidates
df_jobbank
selected_jobbank
occupation_summary
selected_noc
_21
noc21_noc16
noc16_soc
soc_onet
cip_soc
data
_29
temp


In [37]:
# Cell — Build selected occupation mapping with NOC 2021 and NOC 2016 classifications

selected_mapping = (
    selected_jobbank[
        [
            "job_title",
            "noc_2021_code",
            "noc_2021_name",
            "noc_2016_code",
            "noc_2016_name"
        ]
    ]
    .drop_duplicates()
    .copy()
)

print(
    "Selected occupation mapping shape:",
    selected_mapping.shape
)

display(
    selected_mapping
)

Selected occupation mapping shape: (18, 5)


,job_title,noc_2021_code,noc_2021_name,noc_2016_code,noc_2016_name
11,delivery driver,75201.0,Delivery service drivers and door-to-door distributors,7514.0,Delivery and courier service drivers
79,restaurant manager,60030.0,Restaurant and food service managers,631.0,Restaurant and food service managers
95,food service supervisor,62020.0,Food service supervisors,6311.0,Food service supervisors
109,administrative assistant,13110.0,Administrative assistants,1241.0,Administrative assistants
136,"driver, truck",73300.0,Transport truck drivers,7511.0,Transport truck drivers
147,office administrator,13100.0,Administrative officers,1221.0,Administrative officers
200,retail sales associate,64100.0,Retail salespersons and visual merchandisers,6421.0,Retail salespersons
360,licensed practical nurse (L.P.N.),32101.0,Licensed practical nurses,3233.0,Licensed practical nurses
362,continuing care assistant,33102.0,"Nurse aides, orderlies and patient service associates",3413.0,"Nurse aides, orderlies and patient service associates"
379,bookkeeper,12200.0,Accounting technicians and bookkeepers,1311.0,Accounting technicians and bookkeepers


In [38]:
# Cell — Inspect selected occupations and their NOC 2021 classifications

print(
    "selected_jobbank shape:",
    selected_jobbank.shape
)

display(
    selected_jobbank[
        [
            "job_title",
            "noc_2021_code",
            "noc_2021_name"
        ]
    ]
    .drop_duplicates()
    .sort_values("job_title")
)

selected_jobbank shape: (5629, 66)


,job_title,noc_2021_code,noc_2021_name
16515,Administrative Assistant,13110.0,Administrative assistants
11792,Food service supervisor,62020.0,Food service supervisors
22012,Retail sales associate,64100.0,Retail salespersons and visual merchandisers
109,administrative assistant,13110.0,Administrative assistants
379,bookkeeper,12200.0,Accounting technicians and bookkeepers
362,continuing care assistant,33102.0,"Nurse aides, orderlies and patient service associates"
11,delivery driver,75201.0,Delivery service drivers and door-to-door distributors
136,"driver, truck",73300.0,Transport truck drivers
95,food service supervisor,62020.0,Food service supervisors
584,information technology (IT) analyst,21222.0,Information systems specialists


In [39]:
# Check the unique occupations represented in selected_jobbank

print("Number of rows:", len(selected_jobbank))

print("\nUnique NOC 2021 codes:")
print(selected_jobbank["noc_2021_code"].dropna().unique())

print("\nUnique NOC 2021 occupation names:")
print(selected_jobbank["noc_2021_code"].dropna().unique())

print("\nUnique job titles:")
print(selected_jobbank["job_title"].dropna().unique())

Number of rows: 5629

Unique NOC 2021 codes:
[75201. 60030. 62020. 13110. 73300. 13100. 64100. 32101. 33102. 12200.
 21222. 21232. 41220.]

Unique NOC 2021 occupation names:
[75201. 60030. 62020. 13110. 73300. 13100. 64100. 32101. 33102. 12200.
 21222. 21232. 41220.]

Unique job titles:
['delivery driver' 'restaurant manager' 'food service supervisor'
 'administrative assistant' 'driver, truck' 'office administrator'
 'retail sales associate' 'licensed practical nurse (L.P.N.)'
 'continuing care assistant' 'bookkeeper' 'inside sales representative'
 'information technology (IT) analyst' 'office manager'
 'software developer' 'secondary school teacher' 'Food service supervisor'
 'Administrative Assistant' 'Retail sales associate']


In [40]:
# Cell — Create the selected occupation dataframe

df_selected = selected_jobbank.copy()

print("Selected occupation records:", df_selected.shape)
print()

print("NOC 2021 codes:")
print(df_selected["noc_2021_code"].unique())

print()

print("NOC 2021 occupation names:")
print(df_selected["noc_2021_name"].unique())

Selected occupation records: (5629, 66)

NOC 2021 codes:
[75201. 60030. 62020. 13110. 73300. 13100. 64100. 32101. 33102. 12200.
 21222. 21232. 41220.]

NOC 2021 occupation names:
['Delivery service drivers and door-to-door distributors'
 'Restaurant and food service managers' 'Food service supervisors'
 'Administrative assistants' 'Transport truck drivers'
 'Administrative officers' 'Retail salespersons and visual merchandisers'
 'Licensed practical nurses'
 'Nurse aides, orderlies and patient service associates'
 'Accounting technicians and bookkeepers'
 'Information systems specialists' 'Software developers and programmers'
 'Secondary school teachers']


In [41]:
# Cell — Filter Job Bank data to official Week 4 NOC selections

selected_noc_codes = [
    "21232",  # Software developers and programmers
    "21222",  # Information systems specialists
    "13110",  # Administrative assistants
    "12200",  # Accounting technicians and bookkeepers
    "13100",  # Administrative officers
    "60030",  # Restaurant and food service managers
    "62020",  # Food service supervisors
    "64100",  # Retail salespersons and visual merchandisers
    "73300",  # Transport truck drivers
    "75201",  # Delivery service drivers and door-to-door distributors
    "41220",  # Secondary school teachers
    "33102",  # Nurse aides, orderlies and patient service associates
    "32101"   # Licensed practical nurses
]


selected_occupations = (
    df_jobbank[
        df_jobbank["noc_2021_code"].astype(str).isin(
            selected_noc_codes
        )
    ]
    .copy()
)


print(
    "Selected occupation records:",
    selected_occupations.shape
)

print(
    "\nSelected NOC 2021 classifications:"
)

display(
    selected_occupations[
        [
            "noc_2021_code",
            "noc_2021_name"
        ]
    ]
    .drop_duplicates()
    .sort_values("noc_2021_code")
)

Selected occupation records: (0, 66)

Selected NOC 2021 classifications:


,noc_2021_code,noc_2021_name


In [42]:
# Cell — Inspect NOC 2021 to NOC 2016 crosswalk

print("NOC 2021 → NOC 2016 columns:")
print(noc21_noc16.columns.tolist())

print("\nNumber of rows:", len(noc21_noc16))

NOC 2021 → NOC 2016 columns:
['NOC 2016 V1.3 Code', 'NOC 2016 V1.3 Title', 'GSIM Type of Change', 'NOC 2021 V1.0 Code', 'NOC 2021 V1.0 Title', 'Notes', 'Unnamed: 6']

Number of rows: 585


In [43]:
# Cell — Standardize NOC 2021 and NOC 2016 codes for mapping

noc21_noc16["NOC 2021 V1.0 Code"] = (
    pd.to_numeric(
        noc21_noc16["NOC 2021 V1.0 Code"],
        errors="coerce"
    )
)

noc21_noc16["NOC 2016 V1.3 Code"] = (
    pd.to_numeric(
        noc21_noc16["NOC 2016 V1.3 Code"],
        errors="coerce"
    )
)

print("NOC 2021 codes available:",
      noc21_noc16["NOC 2021 V1.0 Code"].notna().sum())

print("NOC 2016 codes available:",
      noc21_noc16["NOC 2016 V1.3 Code"].notna().sum())

NOC 2021 codes available: 585
NOC 2016 codes available: 585


In [44]:
# Cell — Map selected NOC 2021 codes to NOC 2016

noc21_to_noc16_selected = noc21_noc16[
    noc21_noc16["NOC 2021 V1.0 Code"].isin(selected_noc_codes)
].copy()

display(
    noc21_to_noc16_selected[
        [
            "NOC 2021 V1.0 Code",
            "NOC 2021 V1.0 Title",
            "NOC 2016 V1.3 Code",
            "NOC 2016 V1.3 Title",
            "GSIM Type of Change"
        ]
    ]
)

,NOC 2021 V1.0 Code,NOC 2021 V1.0 Title,NOC 2016 V1.3 Code,NOC 2016 V1.3 Title,GSIM Type of Change


In [45]:
# Cell — Create validated NOC 2021 to NOC 2016 mapping

validated_noc_mapping = pd.DataFrame({

    "noc21_code": [
        "12200",
        "13100",
        "13110",
        "21222",
        "21232",
        "32101",
        "33102",
        "41220",
        "60030",
        "62020",
        "64100",
        "73300",
        "75201"
    ],

    "noc2016_code": [
        "1311",
        "1221",
        "1241",
        "2171",
        "2174",
        "3233",
        "3413",
        "4031",
        "0631",
        "6311",
        "6421",
        "7511",
        "7514"
    ],

    "mapping_note": [
        "Direct code/name change",
        "Direct code/name change",
        "Selected relevant mapping; excluded HR manager transfer",
        "Split-off mapping",
        "Breakdown mapping",
        "Direct code/name change",
        "Direct code/name change",
        "Direct code/name change",
        "Direct code/name change",
        "Direct code/name change",
        "Selected relevant retail mapping; excluded creative designer transfer",
        "Direct code/name change",
        "Relevant delivery/courier mapping"
    ]
})


display(
    validated_noc_mapping
)

,noc21_code,noc2016_code,mapping_note
0,12200,1311,Direct code/name change
1,13100,1221,Direct code/name change
2,13110,1241,Selected relevant mapping; excluded HR manager transfer
3,21222,2171,Split-off mapping
4,21232,2174,Breakdown mapping
5,32101,3233,Direct code/name change
6,33102,3413,Direct code/name change
7,41220,4031,Direct code/name change
8,60030,0631,Direct code/name change
9,62020,6311,Direct code/name change


In [46]:
# Cell — Validate NOC 2021 mapping coverage

print("Selected occupations:", len(selected_noc_codes))
print("Validated mappings:", len(validated_noc_mapping))

missing_mappings = set(selected_noc_codes) - set(
    validated_noc_mapping["noc21_code"]
)

print("\nMissing NOC 2021 mappings:")
print(missing_mappings)

Selected occupations: 13
Validated mappings: 13

Missing NOC 2021 mappings:
set()


In [47]:
# Cell — Load NOC 2016 to SOC 2018 crosswalk

noc16_soc = pd.read_csv(
    NOC16_SOC_FILE,
    encoding="latin1"
)

print(
    "NOC 2016 → SOC 2018 crosswalk loaded successfully."
)

print(
    "Shape:",
    noc16_soc.shape
)

print()

print(
    "Columns:"
)

print(
    noc16_soc.columns.tolist()
)

NOC 2016 → SOC 2018 crosswalk loaded successfully.
Shape: (1250, 6)

Columns:
['NOC 2016  Version 1.3 Code', 'NOC 2016  Version 1.3 Title', 'Partial', 'SOC 2018 (US) Code', 'SOC 2018 (US) Title', 'Explanatory Notes']


In [48]:
# Cell — Clean NOC 2016 to SOC 2018 column names

noc16_soc.columns = (
    noc16_soc.columns
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

print(noc16_soc.columns.tolist())

['NOC 2016 Version 1.3 Code', 'NOC 2016 Version 1.3 Title', 'Partial', 'SOC 2018 (US) Code', 'SOC 2018 (US) Title', 'Explanatory Notes']


In [49]:
# Cell — Inspect NOC 2016 to SOC 2018 crosswalk records

display(noc16_soc.head(10))

,NOC 2016 Version 1.3 Code,NOC 2016 Version 1.3 Title,Partial,SOC 2018 (US) Code,SOC 2018 (US) Title,Explanatory Notes
0,11,Legislators,NaN,11-1031,Legislators,NaN
1,12,Senior government managers and officials,NaN,11-1011,Chief Executives,NaN
2,13,"Senior managers - financial, communications and other business services",NaN,11-1011,Chief Executives,NaN
3,14,"Senior managers - health, education, social and community services and membership organizations",*,11-1011,Chief Executives,Only highest level management positions
4,14,"Senior managers - health, education, social and community services and membership organizations",*,11-1021,General and Operations Managers,Only general and operations managers as specified in this NOC code
5,14,"Senior managers - health, education, social and community services and membership organizations",*,11-9111,Medical and Health Services Managers,Only administrators and executive directors of hospitals
6,15,"Senior managers - trade, broadcasting and other services, n.e.c.",*,11-1011,Chief Executives,Only highest level management positions
7,15,"Senior managers - trade, broadcasting and other services, n.e.c.",*,11-1021,General and Operations Managers,Only general and operations managers as specified in this NOC code
8,16,"Senior managers - construction, transportation, production and utilities",NaN,11-1011,Chief Executives,NaN
9,111,Financial managers,NaN,11-3031,Financial Managers,NaN


In [50]:
# Cell — Check selected NOC table columns

print(
    selected_noc.columns.tolist()
)

display(
    selected_noc.head()
)

['selected_occupation', 'noc_2021_code']


,selected_occupation,noc_2021_code
0,Software Developer,21232
1,Information Technology (IT) Analyst,21222
2,Administrative Assistant,13110
3,Bookkeeper,12200
4,Office Administrator,13100


In [51]:
# Cell — Create clean selected occupation and NOC table

selected_occupation_noc = (
    selected_noc[
        [
            "selected_occupation",
            "noc_2021_code"
        ]
    ]
    .copy()
)

# Keep NOC 2021 codes as strings for crosswalk matching
selected_occupation_noc["noc_2021_code"] = (
    selected_occupation_noc["noc_2021_code"]
    .astype(str)
    .str.strip()
)

print(
    "Selected occupation labels:",
    len(selected_occupation_noc)
)

print(
    "Unique NOC 2021 codes:",
    selected_occupation_noc["noc_2021_code"].nunique()
)

display(
    selected_occupation_noc
)

Selected occupation labels: 15
Unique NOC 2021 codes: 13


,selected_occupation,noc_2021_code
0,Software Developer,21232
1,Information Technology (IT) Analyst,21222
2,Administrative Assistant,13110
3,Bookkeeper,12200
4,Office Administrator,13100
5,Office Manager,13100
6,Restaurant Manager,60030
7,Food Service Supervisor,62020
8,Retail Sales Associate,64100
9,Inside Sales Representative,64100


In [52]:
# Cell — Add validated NOC 2016 mappings to selected occupations

occupation_noc_mapping = selected_occupation_noc.merge(
    validated_noc_mapping,
    left_on="noc_2021_code",
    right_on="noc21_code",
    how="left"
)

print(
    "Shape:",
    occupation_noc_mapping.shape
)

display(
    occupation_noc_mapping
)

Shape: (15, 5)


,selected_occupation,noc_2021_code,noc21_code,noc2016_code,mapping_note
0,Software Developer,21232,21232,2174,Breakdown mapping
1,Information Technology (IT) Analyst,21222,21222,2171,Split-off mapping
2,Administrative Assistant,13110,13110,1241,Selected relevant mapping; excluded HR manager transfer
3,Bookkeeper,12200,12200,1311,Direct code/name change
4,Office Administrator,13100,13100,1221,Direct code/name change
5,Office Manager,13100,13100,1221,Direct code/name change
6,Restaurant Manager,60030,60030,0631,Direct code/name change
7,Food Service Supervisor,62020,62020,6311,Direct code/name change
8,Retail Sales Associate,64100,64100,6421,Selected relevant retail mapping; excluded creative designer transfer
9,Inside Sales Representative,64100,64100,6421,Selected relevant retail mapping; excluded creative designer transfer


In [53]:
# Cell — Mapping completeness check

missing = occupation_noc_mapping[
    occupation_noc_mapping["noc2016_code"].isna()
]

print("Total selected occupation labels:", len(occupation_noc_mapping))
print("Missing NOC 2016 mappings:", len(missing))

if len(missing) > 0:
    display(missing)
else:
    print("All selected occupation labels have a validated NOC 2016 mapping.")

Total selected occupation labels: 15
Missing NOC 2016 mappings: 0
All selected occupation labels have a validated NOC 2016 mapping.


In [54]:
# Cell — Check NOC 2016 to SOC 2018 columns

print(
    "NOC 2016 to SOC 2018 columns:"
)

print(
    noc16_soc.columns.tolist()
)

NOC 2016 to SOC 2018 columns:
['NOC 2016 Version 1.3 Code', 'NOC 2016 Version 1.3 Title', 'Partial', 'SOC 2018 (US) Code', 'SOC 2018 (US) Title', 'Explanatory Notes']


In [55]:
# Cell — Prepare NOC 2016 codes for SOC 2018 mapping

noc16_soc.columns = (
    noc16_soc.columns
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

noc16_soc["NOC 2016 Version 1.3 Code"] = (
    noc16_soc["NOC 2016 Version 1.3 Code"]
    .astype(str)
    .str.strip()
)

occupation_noc_mapping["noc2016_code"] = (
    occupation_noc_mapping["noc2016_code"]
    .astype(str)
    .str.strip()
)

print(
    "Selected NOC 2016 codes:"
)

print(
    sorted(
        occupation_noc_mapping[
            "noc2016_code"
        ]
        .dropna()
        .unique()
    )
)

Selected NOC 2016 codes:
['0631', '1221', '1241', '1311', '2171', '2174', '3233', '3413', '4031', '6311', '6421', '7511', '7514']


In [56]:
# Cell — Map NOC 2016 occupations to SOC 2018 classifications

occupation_soc_mapping = occupation_noc_mapping.merge(
    noc16_soc,
    left_on="noc2016_code",
    right_on="NOC 2016 Version 1.3 Code",
    how="left"
)

print(
    "Occupation → SOC mapping shape:",
    occupation_soc_mapping.shape
)

display(
    occupation_soc_mapping[
        [
            "selected_occupation",
            "noc_2021_code",
            "noc2016_code",
            "SOC 2018 (US) Code",
            "SOC 2018 (US) Title",
            "Partial"
        ]
    ]
)

Occupation → SOC mapping shape: (38, 11)


,selected_occupation,noc_2021_code,noc2016_code,SOC 2018 (US) Code,SOC 2018 (US) Title,Partial
0,Software Developer,21232,2174,15-1251,Computer Programmers,*
1,Software Developer,21232,2174,15-1252,Software Developers,*
2,Information Technology (IT) Analyst,21222,2171,15-1211,Computer Systems Analysts,*
3,Information Technology (IT) Analyst,21222,2171,15-1212,Information Security Analysts,*
4,Information Technology (IT) Analyst,21222,2171,15-1253,Software Quality Assurance Analysts and Testers,*
5,Administrative Assistant,13110,1241,43-6014,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",NaN
6,Bookkeeper,12200,1311,43-3031,"Bookkeeping, Accounting, and Auditing Clerks",NaN
7,Office Administrator,13100,1221,11-3012,Administrative Services Managers,*
8,Office Administrator,13100,1221,13-2031,Budget Analysts,*
9,Office Administrator,13100,1221,25-9044,"Teaching Assistants, Postsecondary",*


In [57]:
# Cell — SOC mapping completeness

missing_soc = occupation_soc_mapping[
    occupation_soc_mapping["SOC 2018 (US) Code"].isna()
]

print(
    "Occupation → SOC mapping rows:",
    len(occupation_soc_mapping)
)

print(
    "Missing SOC mappings:",
    len(missing_soc)
)

if len(missing_soc) > 0:
    display(
        missing_soc[
            [
                "selected_occupation",
                "noc_2021_code",
                "noc2016_code"
            ]
        ]
    )
else:
    print(
        "All selected occupations have a SOC 2018 mapping."
    )

Occupation → SOC mapping rows: 38
Missing SOC mappings: 1


,selected_occupation,noc_2021_code,noc2016_code
17,Restaurant Manager,60030,0631


In [58]:
# Cell — Check restaurant manager SOC mapping

restaurant_soc = noc16_soc[
    noc16_soc["NOC 2016 Version 1.3 Code"] == "0631"
]

print(
    "Rows found for NOC 0631:",
    len(restaurant_soc)
)

display(
    restaurant_soc
)

Rows found for NOC 0631: 0


,NOC 2016 Version 1.3 Code,NOC 2016 Version 1.3 Title,Partial,SOC 2018 (US) Code,SOC 2018 (US) Title,Explanatory Notes


In [59]:
# Cell — Search related restaurant SOC mappings

restaurant_related = noc16_soc[
    noc16_soc["NOC 2016 Version 1.3 Title"]
    .str.contains(
        "restaurant|food service",
        case=False,
        na=False
    )
]

print(
    "Related restaurant/food-service rows:",
    len(restaurant_related)
)

display(
    restaurant_related
)

Related restaurant/food-service rows: 2


,NOC 2016 Version 1.3 Code,NOC 2016 Version 1.3 Title,Partial,SOC 2018 (US) Code,SOC 2018 (US) Title,Explanatory Notes
70,631,Restaurant and food service managers,NaN,11-9051,Food Service Managers,NaN
719,6311,Food service supervisors,NaN,35-1012,First-Line Supervisors of Food Preparation and Serving Workers,NaN


In [60]:
# Cell — Create NOC 2016 matching keys

noc16_soc["noc2016_match_code"] = (
    noc16_soc["NOC 2016 Version 1.3 Code"]
    .astype(str)
    .str.strip()
    .str.lstrip("0")
)

occupation_noc_mapping["noc2016_match_code"] = (
    occupation_noc_mapping["noc2016_code"]
    .astype(str)
    .str.strip()
    .str.lstrip("0")
)

print(
    "Restaurant Manager mapping key:",
    occupation_noc_mapping.loc[
        occupation_noc_mapping["noc2016_code"] == "0631",
        "noc2016_match_code"
    ].unique()
)

print(
    "Crosswalk restaurant mapping key:",
    noc16_soc.loc[
        noc16_soc["NOC 2016 Version 1.3 Code"] == "631",
        "noc2016_match_code"
    ].unique()
)

Restaurant Manager mapping key: ['631']
Crosswalk restaurant mapping key: ['631']


In [61]:
# Cell — Map NOC 2016 to SOC 2018 using matching keys

occupation_soc_mapping = occupation_noc_mapping.merge(
    noc16_soc,
    left_on="noc2016_match_code",
    right_on="noc2016_match_code",
    how="left"
)

print(
    "Occupation → SOC mapping shape:",
    occupation_soc_mapping.shape
)

display(
    occupation_soc_mapping[
        [
            "selected_occupation",
            "noc_2021_code",
            "noc2016_code",
            "SOC 2018 (US) Code",
            "SOC 2018 (US) Title",
            "Partial"
        ]
    ]
)

Occupation → SOC mapping shape: (38, 12)


,selected_occupation,noc_2021_code,noc2016_code,SOC 2018 (US) Code,SOC 2018 (US) Title,Partial
0,Software Developer,21232,2174,15-1251,Computer Programmers,*
1,Software Developer,21232,2174,15-1252,Software Developers,*
2,Information Technology (IT) Analyst,21222,2171,15-1211,Computer Systems Analysts,*
3,Information Technology (IT) Analyst,21222,2171,15-1212,Information Security Analysts,*
4,Information Technology (IT) Analyst,21222,2171,15-1253,Software Quality Assurance Analysts and Testers,*
5,Administrative Assistant,13110,1241,43-6014,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",NaN
6,Bookkeeper,12200,1311,43-3031,"Bookkeeping, Accounting, and Auditing Clerks",NaN
7,Office Administrator,13100,1221,11-3012,Administrative Services Managers,*
8,Office Administrator,13100,1221,13-2031,Budget Analysts,*
9,Office Administrator,13100,1221,25-9044,"Teaching Assistants, Postsecondary",*


In [62]:
# Cell — Load SOC 2018 → O*NET crosswalk

SOC_ONET_PATH = os.path.join(
    CROSSWALK_PATH,
    "SOC2018_to_ONET",
    "2019_to_SOC_Crosswalk.xlsx"
)

soc_onet = pd.read_excel(
    SOC_ONET_PATH,
    sheet_name="O-NET-SOC 2019 Occupation Listi",
    header=None
)

print("SOC → O*NET crosswalk shape:", soc_onet.shape)
display(soc_onet.head(10))

SOC → O*NET crosswalk shape: (1020, 4)


,0,1,2,3
0,O*NET-SOC 2019 Occupation Listings,NaN,NaN,NaN
1,Crosswalk O*NET-SOC 2019 to 2018 SOC,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN
3,O*NET-SOC 2019 Code,O*NET-SOC 2019 Title,2018 SOC Code,2018 SOC Title
4,11-1011.00,Chief Executives,11-1011,Chief Executives
5,11-1011.03,Chief Sustainability Officers,11-1011,Chief Executives
6,11-1021.00,General and Operations Managers,11-1021,General and Operations Managers
7,11-1031.00,Legislators,11-1031,Legislators
8,11-2011.00,Advertising and Promotions Managers,11-2011,Advertising and Promotions Managers
9,11-2021.00,Marketing Managers,11-2021,Marketing Managers


In [63]:
# Cell — Clean SOC 2018 → O*NET crosswalk

soc_onet = pd.read_excel(
    SOC_ONET_PATH,
    sheet_name="O-NET-SOC 2019 Occupation Listi",
    skiprows=3
)

soc_onet.columns = (
    soc_onet.columns
    .str.strip()
)

print("Shape:", soc_onet.shape)
print()
print("Columns:")
print(soc_onet.columns.tolist())

display(soc_onet.head())

Shape: (1016, 4)

Columns:
['O*NET-SOC 2019 Code', 'O*NET-SOC 2019 Title', '2018 SOC Code', '2018 SOC Title']


,O*NET-SOC 2019 Code,O*NET-SOC 2019 Title,2018 SOC Code,2018 SOC Title
0,11-1011.00,Chief Executives,11-1011,Chief Executives
1,11-1011.03,Chief Sustainability Officers,11-1011,Chief Executives
2,11-1021.00,General and Operations Managers,11-1021,General and Operations Managers
3,11-1031.00,Legislators,11-1031,Legislators
4,11-2011.00,Advertising and Promotions Managers,11-2011,Advertising and Promotions Managers


In [64]:
# Cell — Normalize SOC codes

occupation_soc_mapping["SOC 2018 (US) Code"] = (
    occupation_soc_mapping["SOC 2018 (US) Code"]
    .astype(str)
    .str.strip()
)

soc_onet["2018 SOC Code"] = (
    soc_onet["2018 SOC Code"]
    .astype(str)
    .str.strip()
)

print("Example selected SOC codes:")
print(
    occupation_soc_mapping[
        "SOC 2018 (US) Code"
    ].dropna().unique()[:20]
)

Example selected SOC codes:
['15-1251' '15-1252' '15-1211' '15-1212' '15-1253' '43-6014' '43-3031'
 '11-3012' '13-2031' '25-9044' '43-1011' '43-6011' '11-9051' '35-1012'
 '31-9095' '41-2021' '41-2031' '53-3032' '53-3031' '53-3033']


In [65]:
# Cell — SOC 2018 → O*NET mapping

occupation_onet_mapping = occupation_soc_mapping.merge(
    soc_onet,
    left_on="SOC 2018 (US) Code",
    right_on="2018 SOC Code",
    how="left"
)

print(
    "Occupation → SOC → O*NET shape:",
    occupation_onet_mapping.shape
)

display(
    occupation_onet_mapping[
        [
            "selected_occupation",
            "noc21_code",
            "noc2016_code",
            "SOC 2018 (US) Code",
            "SOC 2018 (US) Title",
            "O*NET-SOC 2019 Code",
            "O*NET-SOC 2019 Title"
        ]
    ]
)

Occupation → SOC → O*NET shape: (39, 16)


,selected_occupation,noc21_code,noc2016_code,SOC 2018 (US) Code,SOC 2018 (US) Title,O*NET-SOC 2019 Code,O*NET-SOC 2019 Title
0,Software Developer,21232,2174,15-1251,Computer Programmers,15-1251.00,Computer Programmers
1,Software Developer,21232,2174,15-1252,Software Developers,15-1252.00,Software Developers
2,Information Technology (IT) Analyst,21222,2171,15-1211,Computer Systems Analysts,15-1211.00,Computer Systems Analysts
3,Information Technology (IT) Analyst,21222,2171,15-1211,Computer Systems Analysts,15-1211.01,Health Informatics Specialists
4,Information Technology (IT) Analyst,21222,2171,15-1212,Information Security Analysts,15-1212.00,Information Security Analysts
5,Information Technology (IT) Analyst,21222,2171,15-1253,Software Quality Assurance Analysts and Testers,15-1253.00,Software Quality Assurance Analysts and Testers
6,Administrative Assistant,13110,1241,43-6014,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",43-6014.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"
7,Bookkeeper,12200,1311,43-3031,"Bookkeeping, Accounting, and Auditing Clerks",43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks"
8,Office Administrator,13100,1221,11-3012,Administrative Services Managers,11-3012.00,Administrative Services Managers
9,Office Administrator,13100,1221,13-2031,Budget Analysts,13-2031.00,Budget Analysts


In [66]:
# Cell — O*NET mapping completeness

missing_onet = occupation_onet_mapping[
    occupation_onet_mapping["O*NET-SOC 2019 Code"].isna()
]

print(
    "Total occupation-SOC records:",
    len(occupation_onet_mapping)
)

print(
    "Records with O*NET mapping:",
    occupation_onet_mapping["O*NET-SOC 2019 Code"].notna().sum()
)

print(
    "Records missing O*NET mapping:",
    len(missing_onet)
)

if len(missing_onet) > 0:
    display(
        missing_onet[
            [
                "selected_occupation",
                "SOC 2018 (US) Code",
                "SOC 2018 (US) Title"
            ]
        ]
    )

Total occupation-SOC records: 39
Records with O*NET mapping: 39
Records missing O*NET mapping: 0


In [67]:
# Cell — Map SOC 2018 → O*NET

occupation_onet_mapping = occupation_soc_mapping.merge(
    soc_onet,
    left_on="SOC 2018 (US) Code",
    right_on="2018 SOC Code",
    how="left"
)

print(
    "Occupation → SOC → O*NET shape:",
    occupation_onet_mapping.shape
)

display(
    occupation_onet_mapping[
        [
            "selected_occupation",
            "noc_2021_code",
            "noc2016_code",
            "SOC 2018 (US) Code",
            "SOC 2018 (US) Title",
            "O*NET-SOC 2019 Code",
            "O*NET-SOC 2019 Title"
        ]
    ]
)

Occupation → SOC → O*NET shape: (39, 16)


,selected_occupation,noc_2021_code,noc2016_code,SOC 2018 (US) Code,SOC 2018 (US) Title,O*NET-SOC 2019 Code,O*NET-SOC 2019 Title
0,Software Developer,21232,2174,15-1251,Computer Programmers,15-1251.00,Computer Programmers
1,Software Developer,21232,2174,15-1252,Software Developers,15-1252.00,Software Developers
2,Information Technology (IT) Analyst,21222,2171,15-1211,Computer Systems Analysts,15-1211.00,Computer Systems Analysts
3,Information Technology (IT) Analyst,21222,2171,15-1211,Computer Systems Analysts,15-1211.01,Health Informatics Specialists
4,Information Technology (IT) Analyst,21222,2171,15-1212,Information Security Analysts,15-1212.00,Information Security Analysts
5,Information Technology (IT) Analyst,21222,2171,15-1253,Software Quality Assurance Analysts and Testers,15-1253.00,Software Quality Assurance Analysts and Testers
6,Administrative Assistant,13110,1241,43-6014,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",43-6014.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"
7,Bookkeeper,12200,1311,43-3031,"Bookkeeping, Accounting, and Auditing Clerks",43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks"
8,Office Administrator,13100,1221,11-3012,Administrative Services Managers,11-3012.00,Administrative Services Managers
9,Office Administrator,13100,1221,13-2031,Budget Analysts,13-2031.00,Budget Analysts


In [68]:
# Cell — O*NET mapping completeness

missing_onet = occupation_onet_mapping[
    occupation_onet_mapping["O*NET-SOC 2019 Code"].isna()
]

print("Total occupation-SOC records:", len(occupation_onet_mapping))

print(
    "Records with O*NET mapping:",
    occupation_onet_mapping["O*NET-SOC 2019 Code"].notna().sum()
)

print(
    "Records missing O*NET mapping:",
    len(missing_onet)
)

if len(missing_onet) > 0:
    display(
        missing_onet[
            [
                "selected_occupation",
                "SOC 2018 (US) Code",
                "SOC 2018 (US) Title"
            ]
        ]
    )
else:
    print("All occupation-SOC records have an O*NET mapping.")

Total occupation-SOC records: 39
Records with O*NET mapping: 39
Records missing O*NET mapping: 0
All occupation-SOC records have an O*NET mapping.


In [69]:
# Cell — Inspect O*NET mapping coverage by selected occupation

onet_mapping_summary = (
    occupation_onet_mapping
    .groupby("selected_occupation")
    .agg(
        noc21_code=("noc_2021_code", "first"),
        soc_count=("SOC 2018 (US) Code", "nunique"),
        onet_count=("O*NET-SOC 2019 Code", "nunique")
    )
    .reset_index()
)

display(onet_mapping_summary)

,selected_occupation,noc21_code,soc_count,onet_count
0,Administrative Assistant,13110,1,1
1,Bookkeeper,12200,1,1
2,Continuing Care Assistant,33102,4,4
3,Delivery Driver,75201,2,2
4,"Driver, Truck",73300,1,1
5,Food Service Supervisor,62020,1,1
6,Information Technology (IT) Analyst,21222,3,4
7,Inside Sales Representative,64100,3,3
8,Licensed Practical Nurse (L.P.N.),32101,2,2
9,Office Administrator,13100,5,5


In [70]:
# Cell — Identify one-to-many mappings

one_to_many_onet = onet_mapping_summary[
    onet_mapping_summary["onet_count"] > 1
].copy()

print(
    "Selected occupations with multiple O*NET mappings:",
    len(one_to_many_onet)
)

display(one_to_many_onet)

Selected occupations with multiple O*NET mappings: 10


,selected_occupation,noc21_code,soc_count,onet_count
2,Continuing Care Assistant,33102,4,4
3,Delivery Driver,75201,2,2
6,Information Technology (IT) Analyst,21222,3,4
7,Inside Sales Representative,64100,3,3
8,Licensed Practical Nurse (L.P.N.),32101,2,2
9,Office Administrator,13100,5,5
10,Office Manager,13100,5,5
12,Retail Sales Associate,64100,3,3
13,Secondary School Teacher,41220,4,4
14,Software Developer,21232,2,2


In [71]:
# Cell — Inspect one-to-many O*NET mappings

one_to_many_onet = onet_mapping_summary[
    onet_mapping_summary["onet_count"] > 1
]

display(
    occupation_onet_mapping[
        occupation_onet_mapping["selected_occupation"].isin(
            one_to_many_onet["selected_occupation"]
        )
    ][
        [
            "selected_occupation",
            "noc_2021_code",
            "noc2016_code",
            "SOC 2018 (US) Code",
            "SOC 2018 (US) Title",
            "O*NET-SOC 2019 Code",
            "O*NET-SOC 2019 Title"
        ]
    ]
    .sort_values(
        ["selected_occupation", "SOC 2018 (US) Code"]
    )
)

,selected_occupation,noc_2021_code,noc2016_code,SOC 2018 (US) Code,SOC 2018 (US) Title,O*NET-SOC 2019 Code,O*NET-SOC 2019 Title
33,Continuing Care Assistant,33102,3413,31-1122,Personal Care Aides,31-1122.00,Personal Care Aides
34,Continuing Care Assistant,33102,3413,31-1131,Nursing Assistants,31-1131.00,Nursing Assistants
35,Continuing Care Assistant,33102,3413,31-1132,Orderlies,31-1132.00,Orderlies
36,Continuing Care Assistant,33102,3413,31-1133,Psychiatric Aides,31-1133.00,Psychiatric Aides
27,Delivery Driver,75201,7514,53-3031,Driver/Sales Workers,53-3031.00,Driver/Sales Workers
28,Delivery Driver,75201,7514,53-3033,Light Truck Drivers,53-3033.00,Light Truck Drivers
2,Information Technology (IT) Analyst,21222,2171,15-1211,Computer Systems Analysts,15-1211.00,Computer Systems Analysts
3,Information Technology (IT) Analyst,21222,2171,15-1211,Computer Systems Analysts,15-1211.01,Health Informatics Specialists
4,Information Technology (IT) Analyst,21222,2171,15-1212,Information Security Analysts,15-1212.00,Information Security Analysts
5,Information Technology (IT) Analyst,21222,2171,15-1253,Software Quality Assurance Analysts and Testers,15-1253.00,Software Quality Assurance Analysts and Testers


In [72]:
# Cell — Curate validated SOC mappings

validated_soc_codes = {
    "Software Developer": ["15-1252"],
    "Information Technology (IT) Analyst": ["15-1211", "15-1212", "15-1253"],
    "Administrative Assistant": ["43-6014"],
    "Bookkeeper": ["43-3031"],
    "Office Administrator": ["43-1011", "43-6011"],
    "Office Manager": ["43-1011"],
    "Restaurant Manager": ["11-9051"],
    "Food Service Supervisor": ["35-1012"],
    "Retail Sales Associate": ["41-2031"],
    "Inside Sales Representative": ["41-2031"],
    "Driver, Truck": ["53-3032"],
    "Delivery Driver": ["53-3033"],
    "Secondary School Teacher": ["25-2031"],
    "Continuing Care Assistant": ["31-1131", "31-1132"],
    "Licensed Practical Nurse (L.P.N.)": ["29-2061"],
}

In [73]:
validated_onet_mapping = occupation_onet_mapping[
    occupation_onet_mapping.apply(
        lambda row: row["SOC 2018 (US) Code"]
        in validated_soc_codes.get(row["selected_occupation"], []),
        axis=1
    )
].copy()

print("Validated occupation → SOC → O*NET records:",
      len(validated_onet_mapping))

display(
    validated_onet_mapping[
        [
            "selected_occupation",
            "noc21_code",
            "noc2016_code",
            "SOC 2018 (US) Code",
            "SOC 2018 (US) Title",
            "O*NET-SOC 2019 Code",
            "O*NET-SOC 2019 Title"
        ]
    ]
)

Validated occupation → SOC → O*NET records: 20


,selected_occupation,noc21_code,noc2016_code,SOC 2018 (US) Code,SOC 2018 (US) Title,O*NET-SOC 2019 Code,O*NET-SOC 2019 Title
1,Software Developer,21232,2174,15-1252,Software Developers,15-1252.00,Software Developers
2,Information Technology (IT) Analyst,21222,2171,15-1211,Computer Systems Analysts,15-1211.00,Computer Systems Analysts
3,Information Technology (IT) Analyst,21222,2171,15-1211,Computer Systems Analysts,15-1211.01,Health Informatics Specialists
4,Information Technology (IT) Analyst,21222,2171,15-1212,Information Security Analysts,15-1212.00,Information Security Analysts
5,Information Technology (IT) Analyst,21222,2171,15-1253,Software Quality Assurance Analysts and Testers,15-1253.00,Software Quality Assurance Analysts and Testers
6,Administrative Assistant,13110,1241,43-6014,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",43-6014.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive"
7,Bookkeeper,12200,1311,43-3031,"Bookkeeping, Accounting, and Auditing Clerks",43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks"
11,Office Administrator,13100,1221,43-1011,First-Line Supervisors of Office and Administrative Support Workers,43-1011.00,First-Line Supervisors of Office and Administrative Support Workers
12,Office Administrator,13100,1221,43-6011,Executive Secretaries and Executive Administrative Assistants,43-6011.00,Executive Secretaries and Executive Administrative Assistants
16,Office Manager,13100,1221,43-1011,First-Line Supervisors of Office and Administrative Support Workers,43-1011.00,First-Line Supervisors of Office and Administrative Support Workers


In [74]:
# Cell — Validate curated SOC mapping coverage

print(
    "Selected occupation labels:",
    len(validated_soc_codes)
)

print(
    "Occupation labels with validated SOC mappings:",
    validated_onet_mapping["selected_occupation"].nunique()
)

missing_validated = set(validated_soc_codes) - set(
    validated_onet_mapping["selected_occupation"]
)

print("Missing validated mappings:")
print(missing_validated)

Selected occupation labels: 15
Occupation labels with validated SOC mappings: 15
Missing validated mappings:
set()


In [75]:
# Cell — Save validated SOC → O*NET mapping

TABLES_PATH = os.path.join(
    OUTPUT_PATH,
    "Tables"
)

os.makedirs(
    TABLES_PATH,
    exist_ok=True
)

validated_mapping_path = os.path.join(
    TABLES_PATH,
    "validated_occupation_onet_mapping.csv"
)

validated_onet_mapping.to_csv(
    validated_mapping_path,
    index=False
)

print("Validated mapping saved to:")
print(validated_mapping_path)

print("\nRecords:", len(validated_onet_mapping))
print(
    "Occupations:",
    validated_onet_mapping["selected_occupation"].nunique()
)

Validated mapping saved to:
C:\Users\Admin\Capstone_Project\Outputs\Tables\validated_occupation_onet_mapping.csv

Records: 20
Occupations: 15


In [76]:
# Cell — Create validated occupation mapping summary

validation_summary = (
    validated_onet_mapping
    .groupby("selected_occupation")
    .agg(
        NOC_2021=("noc_2021_code", "first"),
        NOC_2016=("noc2016_code", "first"),
        SOC_count=("SOC 2018 (US) Code", "nunique"),
        ONET_count=("O*NET-SOC 2019 Code", "nunique")
    )
    .reset_index()
    .sort_values("selected_occupation")
)

display(validation_summary)

,selected_occupation,NOC_2021,NOC_2016,SOC_count,ONET_count
0,Administrative Assistant,13110,1241,1,1
1,Bookkeeper,12200,1311,1,1
2,Continuing Care Assistant,33102,3413,2,2
3,Delivery Driver,75201,7514,1,1
4,"Driver, Truck",73300,7511,1,1
5,Food Service Supervisor,62020,6311,1,1
6,Information Technology (IT) Analyst,21222,2171,3,4
7,Inside Sales Representative,64100,6421,1,1
8,Licensed Practical Nurse (L.P.N.),32101,3233,1,1
9,Office Administrator,13100,1221,2,2


In [77]:
# Cell — Inspect O*NET identifiers

print("Occupation table:")
print(occupation.columns.tolist())

print("\nSkills table:")
print(skills.columns.tolist())

print("\nKnowledge table:")
print(knowledge.columns.tolist())

print("\nAbilities table:")
print(abilities.columns.tolist())

print("\nTasks table:")
print(tasks.columns.tolist())

print("\nEducation table:")
print(education.columns.tolist())

print("\nJob Zones table:")
print(job_zones.columns.tolist())

print("\nTraining table:")
print(training.columns.tolist())

Occupation table:
['O*NET-SOC Code', 'Title', 'Description']

Skills table:
['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']

Knowledge table:
['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']

Abilities table:
['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'Data Value', 'N', 'Standard Error', 'Lower CI Bound', 'Upper CI Bound', 'Recommend Suppress', 'Not Relevant', 'Date', 'Domain Source']

Tasks table:
['O*NET-SOC Code', 'Title', 'Task ID', 'Task', 'Task Type', 'Incumbents Responding', 'Date', 'Domain Source']

Education table:
['O*NET-SOC Code', 'Title', 'Element ID', 'Element Name', 'Scale ID', 'Scale Name', 'C

In [78]:
# Cell — Preview the key O*NET datasets

display(occupation.head(3))
display(skills.head(3))
display(knowledge.head(3))
display(abilities.head(3))

,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide overall direction of companies or private and publi...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, shareholders, customers, and employees to address su..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of public or private sector organizations, overseeing..."


,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,IM,Importance,4.12,8,0.125,3.8800,4.3700,N,NaN,08/2023,Analyst
1,11-1011.00,Chief Executives,2.A.1.a,Reading Comprehension,LV,Level,4.62,8,0.183,4.2664,4.9836,N,N,08/2023,Analyst
2,11-1011.00,Chief Executives,2.A.1.b,Active Listening,IM,Importance,4.00,8,0.000,4.0000,4.0000,N,NaN,08/2023,Analyst


,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,2.C.1.a,Administration and Management,IM,Importance,4.78,28.0,0.1102,4.5564,5.0000,N,NaN,08/2023,Incumbent
1,11-1011.00,Chief Executives,2.C.1.a,Administration and Management,LV,Level,6.50,28.0,0.2130,6.0666,6.9409,N,N,08/2023,Incumbent
2,11-1011.00,Chief Executives,2.C.1.b,Administrative,IM,Importance,2.42,28.0,0.4651,1.4662,3.3749,N,NaN,08/2023,Incumbent


,O*NET-SOC Code,Title,Element ID,Element Name,Scale ID,Scale Name,Data Value,N,Standard Error,Lower CI Bound,Upper CI Bound,Recommend Suppress,Not Relevant,Date,Domain Source
0,11-1011.00,Chief Executives,1.A.1.a.1,Oral Comprehension,IM,Importance,4.62,8,0.1830,4.2664,4.9836,N,NaN,08/2023,Analyst
1,11-1011.00,Chief Executives,1.A.1.a.1,Oral Comprehension,LV,Level,4.88,8,0.1250,4.6300,5.1200,N,N,08/2023,Analyst
2,11-1011.00,Chief Executives,1.A.1.a.2,Written Comprehension,IM,Importance,4.25,8,0.1637,3.9292,4.5708,N,NaN,08/2023,Analyst


In [79]:
# Cell — Inspect O*NET occupation identifiers

display(
    occupation[
        [col for col in occupation.columns]
    ].head(10)
)

,O*NET-SOC Code,Title,Description
0,11-1011.00,Chief Executives,Determine and formulate policies and provide overall direction of companies or private and publi...
1,11-1011.03,Chief Sustainability Officers,"Communicate and coordinate with management, shareholders, customers, and employees to address su..."
2,11-1021.00,General and Operations Managers,"Plan, direct, or coordinate the operations of public or private sector organizations, overseeing..."
3,11-1031.00,Legislators,"Develop, introduce, or enact laws and statutes at the local, tribal, state, or federal level. In..."
4,11-2011.00,Advertising and Promotions Managers,"Plan, direct, or coordinate advertising policies and programs or produce collateral materials, s..."
5,11-2021.00,Marketing Managers,"Plan, direct, or coordinate marketing policies and programs, such as determining the demand for ..."
6,11-2022.00,Sales Managers,"Plan, direct, or coordinate the actual distribution or movement of a product or service to the c..."
7,11-2032.00,Public Relations Managers,"Plan, direct, or coordinate activities designed to create or maintain a favorable public image o..."
8,11-2033.00,Fundraising Managers,"Plan, direct, or coordinate activities to solicit and maintain funds for special projects or non..."
9,11-3012.00,Administrative Services Managers,"Plan, direct, or coordinate one or more administrative services of an organization, such as reco..."


In [80]:
# Cell — Standardize O*NET occupation codes across datasets

validated_onet_mapping["onet_soc_code"] = (
    validated_onet_mapping["O*NET-SOC 2019 Code"]
    .astype(str)
    .str.strip()
)

occupation["onet_soc_code"] = (
    occupation["O*NET-SOC Code"]
    .astype(str)
    .str.strip()
)

skills["onet_soc_code"] = (
    skills["O*NET-SOC Code"]
    .astype(str)
    .str.strip()
)

knowledge["onet_soc_code"] = (
    knowledge["O*NET-SOC Code"]
    .astype(str)
    .str.strip()
)

abilities["onet_soc_code"] = (
    abilities["O*NET-SOC Code"]
    .astype(str)
    .str.strip()
)

tasks["onet_soc_code"] = (
    tasks["O*NET-SOC Code"]
    .astype(str)
    .str.strip()
)

education["onet_soc_code"] = (
    education["O*NET-SOC Code"]
    .astype(str)
    .str.strip()
)

job_zones["onet_soc_code"] = (
    job_zones["O*NET-SOC Code"]
    .astype(str)
    .str.strip()
)

training["onet_soc_code"] = (
    training["O*NET-SOC Code"]
    .astype(str)
    .str.strip()
)

print("O*NET identifiers standardized.")

O*NET identifiers standardized.


In [81]:
# Cell — Validate O*NET codes against occupation_data

validated_codes = set(
    validated_onet_mapping["onet_soc_code"]
)

onet_codes = set(
    occupation["onet_soc_code"]
)

missing_onet_codes = validated_codes - onet_codes

print("Validated O*NET codes:", len(validated_codes))
print("O*NET occupation codes:", len(onet_codes))
print()
print("Validated codes missing from O*NET occupation table:")
print(missing_onet_codes)

Validated O*NET codes: 18
O*NET occupation codes: 1016

Validated codes missing from O*NET occupation table:
set()


In [82]:
# Cell — Build validated O*NET occupation profile

onet_profile = (
    validated_onet_mapping[
        [
            "selected_occupation",
            "noc_2021_code",
            "noc2016_code",
            "SOC 2018 (US) Code",
            "SOC 2018 (US) Title",
            "onet_soc_code"
        ]
    ]
    .drop_duplicates()
    .merge(
        occupation[
            [
                "onet_soc_code",
                "Title",
                "Description"
            ]
        ],
        on="onet_soc_code",
        how="left"
    )
)

onet_profile = onet_profile.rename(
    columns={
        "Title": "onet_title",
        "Description": "onet_description"
    }
)

display(onet_profile)

,selected_occupation,noc_2021_code,noc2016_code,SOC 2018 (US) Code,SOC 2018 (US) Title,onet_soc_code,onet_title,onet_description
0,Software Developer,21232,2174,15-1252,Software Developers,15-1252.00,Software Developers,"Research, design, and develop computer and network software or specialized utility programs. Ana..."
1,Information Technology (IT) Analyst,21222,2171,15-1211,Computer Systems Analysts,15-1211.00,Computer Systems Analysts,"Analyze science, engineering, business, and other data processing problems to develop and implem..."
2,Information Technology (IT) Analyst,21222,2171,15-1211,Computer Systems Analysts,15-1211.01,Health Informatics Specialists,"Apply knowledge of nursing and informatics to assist in the design, development, and ongoing mod..."
3,Information Technology (IT) Analyst,21222,2171,15-1212,Information Security Analysts,15-1212.00,Information Security Analysts,"Plan, implement, upgrade, or monitor security measures for the protection of computer networks a..."
4,Information Technology (IT) Analyst,21222,2171,15-1253,Software Quality Assurance Analysts and Testers,15-1253.00,Software Quality Assurance Analysts and Testers,Develop and execute software tests to identify software problems and their causes. Test system m...
5,Administrative Assistant,13110,1241,43-6014,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",43-6014.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive","Perform routine administrative functions such as drafting correspondence, scheduling appointment..."
6,Bookkeeper,12200,1311,43-3031,"Bookkeeping, Accounting, and Auditing Clerks",43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks","Compute, classify, and record numerical data to keep financial records complete. Perform any com..."
7,Office Administrator,13100,1221,43-1011,First-Line Supervisors of Office and Administrative Support Workers,43-1011.00,First-Line Supervisors of Office and Administrative Support Workers,Directly supervise and coordinate the activities of clerical and administrative support workers.
8,Office Administrator,13100,1221,43-6011,Executive Secretaries and Executive Administrative Assistants,43-6011.00,Executive Secretaries and Executive Administrative Assistants,"Provide high-level administrative support by conducting research, preparing statistical reports,..."
9,Office Manager,13100,1221,43-1011,First-Line Supervisors of Office and Administrative Support Workers,43-1011.00,First-Line Supervisors of Office and Administrative Support Workers,Directly supervise and coordinate the activities of clerical and administrative support workers.


In [83]:
# Cell — Check occupation profile completeness

profile_check = pd.DataFrame({
    "Metric": [
        "Selected occupation labels",
        "Validated O*NET records",
        "Unique O*NET occupations",
        "Missing O*NET titles",
        "Missing O*NET descriptions"
    ],
    "Value": [
        onet_profile["selected_occupation"].nunique(),
        len(onet_profile),
        onet_profile["onet_soc_code"].nunique(),
        onet_profile["onet_title"].isna().sum(),
        onet_profile["onet_description"].isna().sum()
    ]
})

display(profile_check)

,Metric,Value
0,Selected occupation labels,15
1,Validated O*NET records,20
2,Unique O*NET occupations,18
3,Missing O*NET titles,0
4,Missing O*NET descriptions,0


In [84]:
# Cell — Prepare and clean O*NET skills data

skills_clean = skills[
    [
        "onet_soc_code",
        "Element ID",
        "Element Name",
        "Scale ID",
        "Scale Name",
        "Data Value",
        "Not Relevant"
    ]
].copy()

skills_clean = skills_clean.rename(
    columns={
        "Element ID": "element_id",
        "Element Name": "skill",
        "Scale ID": "scale_id",
        "Scale Name": "scale_name",
        "Data Value": "data_value",
        "Not Relevant": "not_relevant"
    }
)

print("Skills records:", len(skills_clean))

display(skills_clean.head(10))

Skills records: 17880


,onet_soc_code,element_id,skill,scale_id,scale_name,data_value,not_relevant
0,11-1011.00,2.A.1.a,Reading Comprehension,IM,Importance,4.12,NaN
1,11-1011.00,2.A.1.a,Reading Comprehension,LV,Level,4.62,N
2,11-1011.00,2.A.1.b,Active Listening,IM,Importance,4.00,NaN
3,11-1011.00,2.A.1.b,Active Listening,LV,Level,4.75,N
4,11-1011.00,2.A.1.c,Writing,IM,Importance,4.12,NaN
5,11-1011.00,2.A.1.c,Writing,LV,Level,4.38,N
6,11-1011.00,2.A.1.d,Speaking,IM,Importance,4.25,NaN
7,11-1011.00,2.A.1.d,Speaking,LV,Level,4.75,N
8,11-1011.00,2.A.1.e,Mathematics,IM,Importance,3.25,NaN
9,11-1011.00,2.A.1.e,Mathematics,LV,Level,3.50,N


In [85]:
# Cell — Separate O*NET skill importance and level measures

skills_importance = skills_clean[
    skills_clean["scale_id"] == "IM"
].copy()

skills_level = skills_clean[
    skills_clean["scale_id"] == "LV"
].copy()

print("Skill importance records:", len(skills_importance))
print("Skill level records:", len(skills_level))

Skill importance records: 8940
Skill level records: 8940


In [86]:
# Cell — Combine O*NET skill importance and level measures

skills_profile = skills_importance[
    [
        "onet_soc_code",
        "element_id",
        "skill",
        "data_value"
    ]
].rename(
    columns={
        "data_value": "importance"
    }
).merge(
    skills_level[
        [
            "onet_soc_code",
            "element_id",
            "data_value"
        ]
    ],
    on=[
        "onet_soc_code",
        "element_id"
    ],
    how="left"
)

skills_profile = skills_profile.rename(
    columns={
        "data_value": "level"
    }
)

display(skills_profile.head(20))

,onet_soc_code,element_id,skill,importance,level
0,11-1011.00,2.A.1.a,Reading Comprehension,4.12,4.62
1,11-1011.00,2.A.1.b,Active Listening,4.00,4.75
2,11-1011.00,2.A.1.c,Writing,4.12,4.38
3,11-1011.00,2.A.1.d,Speaking,4.25,4.75
4,11-1011.00,2.A.1.e,Mathematics,3.25,3.50
5,11-1011.00,2.A.1.f,Science,1.62,0.75
6,11-1011.00,2.A.2.a,Critical Thinking,4.38,4.75
7,11-1011.00,2.A.2.b,Active Learning,3.75,4.50
8,11-1011.00,2.A.2.c,Learning Strategies,3.12,3.75
9,11-1011.00,2.A.2.d,Monitoring,4.00,5.25


In [87]:
# Cell — Attach selected occupation labels to skills

skills_profile = skills_profile.merge(
    onet_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title"
        ]
    ].drop_duplicates(),
    on="onet_soc_code",
    how="inner"
)

display(
    skills_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title",
            "skill",
            "importance",
            "level"
        ]
    ].head(20)
)

,selected_occupation,onet_soc_code,onet_title,skill,importance,level
0,Restaurant Manager,11-9051.00,Food Service Managers,Reading Comprehension,3.75,3.88
1,Restaurant Manager,11-9051.00,Food Service Managers,Active Listening,3.88,3.50
2,Restaurant Manager,11-9051.00,Food Service Managers,Writing,3.00,3.12
3,Restaurant Manager,11-9051.00,Food Service Managers,Speaking,3.88,4.00
4,Restaurant Manager,11-9051.00,Food Service Managers,Mathematics,2.88,2.88
5,Restaurant Manager,11-9051.00,Food Service Managers,Science,1.62,0.62
6,Restaurant Manager,11-9051.00,Food Service Managers,Critical Thinking,3.62,3.75
7,Restaurant Manager,11-9051.00,Food Service Managers,Active Learning,3.12,3.75
8,Restaurant Manager,11-9051.00,Food Service Managers,Learning Strategies,3.12,3.12
9,Restaurant Manager,11-9051.00,Food Service Managers,Monitoring,3.88,3.88


In [88]:
# Cell — Identify top O*NET skills by selected occupation

top_skills = (
    skills_profile
    .sort_values(
        [
            "selected_occupation",
            "importance"
        ],
        ascending=[True, False]
    )
    .groupby("selected_occupation")
    .head(10)
    .reset_index(drop=True)
)

display(
    top_skills[
        [
            "selected_occupation",
            "onet_title",
            "skill",
            "importance",
            "level"
        ]
    ]
)

,selected_occupation,onet_title,skill,importance,level
0,Administrative Assistant,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",Active Listening,4.00,3.75
1,Administrative Assistant,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",Speaking,4.00,3.62
2,Administrative Assistant,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",Reading Comprehension,3.88,3.88
3,Administrative Assistant,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",Writing,3.75,3.50
4,Administrative Assistant,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",Monitoring,3.12,3.25
...,...,...,...,...,...
145,Software Developer,Software Developers,Speaking,3.12,3.62
146,Software Developer,Software Developers,Monitoring,3.00,3.50
147,Software Developer,Software Developers,Mathematics,2.75,3.25
148,Software Developer,Software Developers,Learning Strategies,2.62,3.12


In [89]:
# Cell — Save O*NET skills profile

skills_output_path = os.path.join(
    TABLES_PATH,
    "onet_skills_profile.csv"
)

skills_profile.to_csv(
    skills_output_path,
    index=False
)

print("Saved:")
print(skills_output_path)

Saved:
C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_skills_profile.csv


In [90]:
# Cell — Prepare O*NET knowledge data

knowledge_clean = knowledge[
    [
        "onet_soc_code",
        "Element ID",
        "Element Name",
        "Scale ID",
        "Scale Name",
        "Data Value",
        "Not Relevant"
    ]
].copy()

knowledge_clean = knowledge_clean.rename(
    columns={
        "Element ID": "element_id",
        "Element Name": "knowledge_area",
        "Scale ID": "scale_id",
        "Scale Name": "scale_name",
        "Data Value": "data_value",
        "Not Relevant": "not_relevant"
    }
)

print("Knowledge records:", len(knowledge_clean))

display(knowledge_clean.head(10))

Knowledge records: 59004


,onet_soc_code,element_id,knowledge_area,scale_id,scale_name,data_value,not_relevant
0,11-1011.00,2.C.1.a,Administration and Management,IM,Importance,4.78,NaN
1,11-1011.00,2.C.1.a,Administration and Management,LV,Level,6.50,N
2,11-1011.00,2.C.1.b,Administrative,IM,Importance,2.42,NaN
3,11-1011.00,2.C.1.b,Administrative,LV,Level,2.69,N
4,11-1011.00,2.C.1.c,Economics and Accounting,IM,Importance,4.04,NaN
5,11-1011.00,2.C.1.c,Economics and Accounting,LV,Level,4.98,N
6,11-1011.00,2.C.1.d,Sales and Marketing,IM,Importance,3.81,NaN
7,11-1011.00,2.C.1.d,Sales and Marketing,LV,Level,5.04,N
8,11-1011.00,2.C.1.e,Customer and Personal Service,IM,Importance,4.39,NaN
9,11-1011.00,2.C.1.e,Customer and Personal Service,LV,Level,5.94,N


In [91]:
# Cell — Separate knowledge importance and level

knowledge_importance = knowledge_clean[
    knowledge_clean["scale_id"] == "IM"
].copy()

knowledge_level = knowledge_clean[
    knowledge_clean["scale_id"] == "LV"
].copy()

print("Knowledge importance records:", len(knowledge_importance))
print("Knowledge level records:", len(knowledge_level))

Knowledge importance records: 29502
Knowledge level records: 29502


In [92]:
# Cell — Combine knowledge importance and level

knowledge_profile = knowledge_importance[
    [
        "onet_soc_code",
        "element_id",
        "knowledge_area",
        "data_value"
    ]
].rename(
    columns={
        "data_value": "importance"
    }
).merge(
    knowledge_level[
        [
            "onet_soc_code",
            "element_id",
            "data_value"
        ]
    ],
    on=[
        "onet_soc_code",
        "element_id"
    ],
    how="left"
)

knowledge_profile = knowledge_profile.rename(
    columns={
        "data_value": "level"
    }
)

display(knowledge_profile.head(20))

,onet_soc_code,element_id,knowledge_area,importance,level
0,11-1011.00,2.C.1.a,Administration and Management,4.78,6.50
1,11-1011.00,2.C.1.b,Administrative,2.42,2.69
2,11-1011.00,2.C.1.c,Economics and Accounting,4.04,4.98
3,11-1011.00,2.C.1.d,Sales and Marketing,3.81,5.04
4,11-1011.00,2.C.1.e,Customer and Personal Service,4.39,5.94
5,11-1011.00,2.C.1.f,Personnel and Human Resources,4.48,5.78
6,11-1011.00,2.C.2.a,Production and Processing,2.71,2.92
7,11-1011.00,2.C.2.b,Food Production,1.14,0.40
8,11-1011.00,2.C.3.a,Computers and Electronics,3.82,5.19
9,11-1011.00,2.C.3.b,Engineering and Technology,3.05,3.51


In [93]:
# Cell — Attach selected occupation labels to knowledge

knowledge_profile = knowledge_profile.merge(
    onet_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title"
        ]
    ].drop_duplicates(),
    on="onet_soc_code",
    how="inner"
)

display(
    knowledge_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title",
            "knowledge_area",
            "importance",
            "level"
        ]
    ].head(20)
)

,selected_occupation,onet_soc_code,onet_title,knowledge_area,importance,level
0,Restaurant Manager,11-9051.00,Food Service Managers,Administration and Management,4.07,4.14
1,Restaurant Manager,11-9051.00,Food Service Managers,Administrative,3.28,4.20
2,Restaurant Manager,11-9051.00,Food Service Managers,Economics and Accounting,3.04,3.29
3,Restaurant Manager,11-9051.00,Food Service Managers,Sales and Marketing,3.44,3.35
4,Restaurant Manager,11-9051.00,Food Service Managers,Customer and Personal Service,4.53,4.88
5,Restaurant Manager,11-9051.00,Food Service Managers,Personnel and Human Resources,3.49,3.80
6,Restaurant Manager,11-9051.00,Food Service Managers,Production and Processing,3.21,3.33
7,Restaurant Manager,11-9051.00,Food Service Managers,Food Production,3.95,4.04
8,Restaurant Manager,11-9051.00,Food Service Managers,Computers and Electronics,2.86,3.70
9,Restaurant Manager,11-9051.00,Food Service Managers,Engineering and Technology,1.47,1.06


In [94]:
# Cell — Top O*NET knowledge areas by occupation

top_knowledge = (
    knowledge_profile
    .sort_values(
        [
            "selected_occupation",
            "importance"
        ],
        ascending=[True, False]
    )
    .groupby("selected_occupation")
    .head(10)
    .reset_index(drop=True)
)

display(
    top_knowledge[
        [
            "selected_occupation",
            "onet_title",
            "knowledge_area",
            "importance",
            "level"
        ]
    ]
)

,selected_occupation,onet_title,knowledge_area,importance,level
0,Administrative Assistant,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",Administrative,4.52,5.80
1,Administrative Assistant,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",English Language,4.29,4.25
2,Administrative Assistant,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",Computers and Electronics,3.84,4.56
3,Administrative Assistant,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",Customer and Personal Service,3.76,4.36
4,Administrative Assistant,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",Administration and Management,3.43,3.15
...,...,...,...,...,...
145,Software Developer,Software Developers,Engineering and Technology,2.80,3.17
146,Software Developer,Software Developers,Design,2.70,3.07
147,Software Developer,Software Developers,Telecommunications,2.62,2.91
148,Software Developer,Software Developers,Public Safety and Security,2.57,1.80


In [95]:
# Cell — Save O*NET knowledge profile

knowledge_output_path = os.path.join(
    TABLES_PATH,
    "onet_knowledge_profile.csv"
)

knowledge_profile.to_csv(
    knowledge_output_path,
    index=False
)

print("Saved:")
print(knowledge_output_path)

Saved:
C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_knowledge_profile.csv


In [96]:
# Cell — Prepare O*NET abilities data

abilities_clean = abilities[
    [
        "onet_soc_code",
        "Element ID",
        "Element Name",
        "Scale ID",
        "Scale Name",
        "Data Value",
        "Not Relevant"
    ]
].copy()

abilities_clean = abilities_clean.rename(
    columns={
        "Element ID": "element_id",
        "Element Name": "ability",
        "Scale ID": "scale_id",
        "Scale Name": "scale_name",
        "Data Value": "data_value",
        "Not Relevant": "not_relevant"
    }
)

print("Abilities records:", len(abilities_clean))

display(abilities_clean.head(10))

Abilities records: 92976


,onet_soc_code,element_id,ability,scale_id,scale_name,data_value,not_relevant
0,11-1011.00,1.A.1.a.1,Oral Comprehension,IM,Importance,4.62,NaN
1,11-1011.00,1.A.1.a.1,Oral Comprehension,LV,Level,4.88,N
2,11-1011.00,1.A.1.a.2,Written Comprehension,IM,Importance,4.25,NaN
3,11-1011.00,1.A.1.a.2,Written Comprehension,LV,Level,4.88,N
4,11-1011.00,1.A.1.a.3,Oral Expression,IM,Importance,4.50,NaN
5,11-1011.00,1.A.1.a.3,Oral Expression,LV,Level,4.88,N
6,11-1011.00,1.A.1.a.4,Written Expression,IM,Importance,4.12,NaN
7,11-1011.00,1.A.1.a.4,Written Expression,LV,Level,4.75,N
8,11-1011.00,1.A.1.b.1,Fluency of Ideas,IM,Importance,3.88,NaN
9,11-1011.00,1.A.1.b.1,Fluency of Ideas,LV,Level,4.62,N


In [97]:
# Cell — Separate abilities importance and level

abilities_importance = abilities_clean[
    abilities_clean["scale_id"] == "IM"
].copy()

abilities_level = abilities_clean[
    abilities_clean["scale_id"] == "LV"
].copy()

print("Ability importance records:", len(abilities_importance))
print("Ability level records:", len(abilities_level))

Ability importance records: 46488
Ability level records: 46488


In [98]:
# Cell — Combine ability importance and level

abilities_profile = abilities_importance[
    [
        "onet_soc_code",
        "element_id",
        "ability",
        "data_value"
    ]
].rename(
    columns={
        "data_value": "importance"
    }
).merge(
    abilities_level[
        [
            "onet_soc_code",
            "element_id",
            "data_value"
        ]
    ],
    on=[
        "onet_soc_code",
        "element_id"
    ],
    how="left"
)

abilities_profile = abilities_profile.rename(
    columns={
        "data_value": "level"
    }
)

display(abilities_profile.head(20))

,onet_soc_code,element_id,ability,importance,level
0,11-1011.00,1.A.1.a.1,Oral Comprehension,4.62,4.88
1,11-1011.00,1.A.1.a.2,Written Comprehension,4.25,4.88
2,11-1011.00,1.A.1.a.3,Oral Expression,4.50,4.88
3,11-1011.00,1.A.1.a.4,Written Expression,4.12,4.75
4,11-1011.00,1.A.1.b.1,Fluency of Ideas,3.88,4.62
5,11-1011.00,1.A.1.b.2,Originality,3.75,4.25
6,11-1011.00,1.A.1.b.3,Problem Sensitivity,4.00,4.88
7,11-1011.00,1.A.1.b.4,Deductive Reasoning,4.12,4.75
8,11-1011.00,1.A.1.b.5,Inductive Reasoning,4.00,4.88
9,11-1011.00,1.A.1.b.6,Information Ordering,4.00,4.12


In [99]:
# Cell — Attach selected occupation labels to abilities

abilities_profile = abilities_profile.merge(
    onet_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title"
        ]
    ].drop_duplicates(),
    on="onet_soc_code",
    how="inner"
)

display(
    abilities_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title",
            "ability",
            "importance",
            "level"
        ]
    ].head(20)
)

,selected_occupation,onet_soc_code,onet_title,ability,importance,level
0,Restaurant Manager,11-9051.00,Food Service Managers,Oral Comprehension,3.88,4.00
1,Restaurant Manager,11-9051.00,Food Service Managers,Written Comprehension,3.88,3.75
2,Restaurant Manager,11-9051.00,Food Service Managers,Oral Expression,3.88,4.00
3,Restaurant Manager,11-9051.00,Food Service Managers,Written Expression,3.00,3.50
4,Restaurant Manager,11-9051.00,Food Service Managers,Fluency of Ideas,2.88,3.00
5,Restaurant Manager,11-9051.00,Food Service Managers,Originality,2.88,3.00
6,Restaurant Manager,11-9051.00,Food Service Managers,Problem Sensitivity,3.88,3.88
7,Restaurant Manager,11-9051.00,Food Service Managers,Deductive Reasoning,3.88,3.75
8,Restaurant Manager,11-9051.00,Food Service Managers,Inductive Reasoning,3.00,3.00
9,Restaurant Manager,11-9051.00,Food Service Managers,Information Ordering,3.00,3.00


In [100]:
# Cell — Top O*NET abilities by occupation

top_abilities = (
    abilities_profile
    .sort_values(
        [
            "selected_occupation",
            "importance"
        ],
        ascending=[True, False]
    )
    .groupby("selected_occupation")
    .head(10)
    .reset_index(drop=True)
)

display(
    top_abilities[
        [
            "selected_occupation",
            "onet_title",
            "ability",
            "importance",
            "level"
        ]
    ]
)

,selected_occupation,onet_title,ability,importance,level
0,Administrative Assistant,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",Oral Comprehension,4.00,4.00
1,Administrative Assistant,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",Written Comprehension,4.00,3.88
2,Administrative Assistant,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",Oral Expression,4.00,4.00
3,Administrative Assistant,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",Written Expression,4.00,3.88
4,Administrative Assistant,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",Near Vision,3.88,3.88
...,...,...,...,...,...
145,Software Developer,Software Developers,Near Vision,3.75,3.75
146,Software Developer,Software Developers,Information Ordering,3.62,3.88
147,Software Developer,Software Developers,Written Expression,3.50,4.00
148,Software Developer,Software Developers,Inductive Reasoning,3.50,3.88


In [101]:
# Cell — Save O*NET abilities profile

abilities_output_path = os.path.join(
    TABLES_PATH,
    "onet_abilities_profile.csv"
)

abilities_profile.to_csv(
    abilities_output_path,
    index=False
)

print("Saved:")
print(abilities_output_path)

Saved:
C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_abilities_profile.csv


In [102]:
# Cell — Prepare O*NET task data

tasks_clean = tasks[
    [
        "onet_soc_code",
        "Task ID",
        "Task",
        "Task Type",
        "Incumbents Responding"
    ]
].copy()

tasks_clean = tasks_clean.rename(
    columns={
        "Task ID": "task_id",
        "Task": "task",
        "Task Type": "task_type",
        "Incumbents Responding": "incumbents_responding"
    }
)

print("Task records:", len(tasks_clean))

display(tasks_clean.head(10))

Task records: 18796


,onet_soc_code,task_id,task,task_type,incumbents_responding
0,11-1011.00,8823,"Direct or coordinate an organization's financial or budget activities to fund operations, maximi...",Core,95.0
1,11-1011.00,8824,"Confer with board members, organization officials, or staff members to discuss issues, coordinat...",Core,95.0
2,11-1011.00,8827,"Prepare budgets for approval, including those for funding or implementation of programs.",Core,95.0
3,11-1011.00,8826,"Direct, plan, or implement policies, objectives, or activities of organizations or businesses to...",Core,94.0
4,11-1011.00,8834,"Prepare or present reports concerning activities, expenses, budgets, government statutes or ruli...",Core,95.0
5,11-1011.00,8836,Implement corrective action plans to solve organizational or departmental problems.,Core,94.0
6,11-1011.00,8825,Analyze operations to evaluate performance of a company or its staff in meeting objectives or to...,Core,94.0
7,11-1011.00,8828,"Direct or coordinate activities of businesses or departments concerned with production, pricing,...",Core,94.0
8,11-1011.00,8832,"Direct human resources activities, including the approval of human resource plans or activities,...",Core,94.0
9,11-1011.00,8831,Appoint department heads or managers and assign or delegate responsibilities to them.,Core,93.0


In [103]:
# Cell — Attach selected occupation labels to O*NET tasks

tasks_profile = tasks_clean.merge(
    onet_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title"
        ]
    ].drop_duplicates(),
    on="onet_soc_code",
    how="inner"
)

display(
    tasks_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title",
            "task_id",
            "task",
            "task_type"
        ]
    ].head(20)
)

,selected_occupation,onet_soc_code,onet_title,task_id,task,task_type
0,Restaurant Manager,11-9051.00,Food Service Managers,15199,Count money and make bank deposits.,Core
1,Restaurant Manager,11-9051.00,Food Service Managers,1086,Establish standards for personnel performance and customer service.,Core
2,Restaurant Manager,11-9051.00,Food Service Managers,1085,Keep records required by government agencies regarding sanitation or food subsidies.,Core
3,Restaurant Manager,11-9051.00,Food Service Managers,1082,Schedule staff hours and assign duties.,Core
4,Restaurant Manager,11-9051.00,Food Service Managers,1078,"Investigate and resolve complaints regarding food quality, service, or accommodations.",Core
5,Restaurant Manager,11-9051.00,Food Service Managers,1090,"Maintain food and equipment inventories, and keep inventory records.",Core
6,Restaurant Manager,11-9051.00,Food Service Managers,1089,"Perform some food preparation or service tasks, such as cooking, clearing tables, and serving fo...",Core
7,Restaurant Manager,11-9051.00,Food Service Managers,1081,"Monitor budgets and payroll records, and review financial transactions to ensure that expenditur...",Core
8,Restaurant Manager,11-9051.00,Food Service Managers,1079,"Schedule and receive food and beverage deliveries, checking delivery contents to verify product ...",Core
9,Restaurant Manager,11-9051.00,Food Service Managers,1084,Coordinate assignments of cooking personnel to ensure economical use of food and timely preparat...,Core


In [104]:
# Cell — Task coverage by occupation

task_coverage = (
    tasks_profile
    .groupby("selected_occupation")
    .agg(
        onet_occupations=("onet_soc_code", "nunique"),
        task_count=("task_id", "count")
    )
    .reset_index()
    .sort_values("selected_occupation")
)

display(task_coverage)

,selected_occupation,onet_occupations,task_count
0,Administrative Assistant,1,31
1,Bookkeeper,1,28
2,Continuing Care Assistant,2,55
3,Delivery Driver,1,13
4,"Driver, Truck",1,29
5,Food Service Supervisor,1,26
6,Information Technology (IT) Analyst,4,80
7,Inside Sales Representative,1,24
8,Licensed Practical Nurse (L.P.N.),1,22
9,Office Administrator,2,50


In [105]:
# Cell — Example task inspection

display(
    tasks_profile[
        tasks_profile["selected_occupation"] == "Software Developer"
    ][
        [
            "selected_occupation",
            "onet_title",
            "task_id",
            "task",
            "task_type"
        ]
    ].head(20)
)

,selected_occupation,onet_title,task_id,task,task_type
78,Software Developer,Software Developers,21662,Analyze user needs and software requirements to determine feasibility of design within time and ...,Core
79,Software Developer,Software Developers,21669,"Develop or direct software system testing or validation procedures, programming, or documentation.",Core
80,Software Developer,Software Developers,21664,"Confer with systems analysts, engineers, programmers and others to design systems and to obtain ...",Core
81,Software Developer,Software Developers,21670,"Modify existing software to correct errors, adapt it to new hardware, or upgrade interfaces and ...",Core
82,Software Developer,Software Developers,21673,"Prepare reports or correspondence concerning project specifications, activities, or status.",Core
83,Software Developer,Software Developers,21661,"Analyze information to determine, recommend, and plan installation of a new system or modificati...",Core
84,Software Developer,Software Developers,21676,"Store, retrieve, and manipulate data for analysis of system capabilities and requirements.",Core
85,Software Developer,Software Developers,21667,"Design, develop and modify software systems, using scientific analysis and mathematical models t...",Core
86,Software Developer,Software Developers,21668,Determine system performance standards.,Core
87,Software Developer,Software Developers,21665,"Consult with customers or other departments on project status, proposals, or technical issues, s...",Core


In [106]:
# Cell — Save O*NET tasks profile

tasks_output_path = os.path.join(
    TABLES_PATH,
    "onet_tasks_profile.csv"
)

tasks_profile.to_csv(
    tasks_output_path,
    index=False
)

print("Saved:")
print(tasks_output_path)

Saved:
C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_tasks_profile.csv


In [107]:
# Cell — Inspect O*NET education data

print("Education records:", len(education))

display(
    education[
        [
            "onet_soc_code",
            "Element ID",
            "Element Name",
            "Scale ID",
            "Scale Name",
            "Category",
            "Data Value"
        ]
    ].head(20)
)

Education records: 11100


,onet_soc_code,Element ID,Element Name,Scale ID,Scale Name,Category,Data Value
0,11-1011.00,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),1.0,0.00
1,11-1011.00,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),2.0,4.46
2,11-1011.00,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),3.0,0.00
3,11-1011.00,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),4.0,0.00
4,11-1011.00,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),5.0,5.15
5,11-1011.00,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),6.0,32.29
6,11-1011.00,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),7.0,0.00
7,11-1011.00,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),8.0,45.91
8,11-1011.00,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),9.0,3.94
9,11-1011.00,2.D.1,Required Level of Education,RL,Required Level Of Education (Categories 1-12),10.0,0.55


In [108]:
# Cell — Education elements and categories

print("Education elements:")
print(education["Element Name"].dropna().unique())

print("\nEducation scale IDs:")
print(education["Scale ID"].dropna().unique())

print("\nEducation categories:")
print(
    education["Category"]
    .dropna()
    .unique()
)

Education elements:
['Required Level of Education' 'Job-Related Professional Certification']

Education scale IDs:
['RL' 'IM']

Education categories:
[ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12.]


In [109]:
# Cell — Attach selected occupation labels to education

education_profile = education.merge(
    onet_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title"
        ]
    ].drop_duplicates(),
    on="onet_soc_code",
    how="inner"
)

display(
    education_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title",
            "Element Name",
            "Scale Name",
            "Category",
            "Data Value"
        ]
    ].head(30)
)

,selected_occupation,onet_soc_code,onet_title,Element Name,Scale Name,Category,Data Value
0,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),1.0,16.14
1,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),2.0,29.66
2,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),3.0,21.34
3,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),4.0,4.56
4,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),5.0,18.56
5,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),6.0,9.73
6,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),7.0,0.00
7,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),8.0,0.00
8,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),9.0,0.00
9,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),10.0,0.00


In [110]:
# Cell — Check O*NET education coverage by selected occupation

education_coverage = (
    education_profile
    .groupby("selected_occupation")
    .agg(
        onet_occupations=("onet_soc_code", "nunique"),
        education_records=("Category", "count")
    )
    .reset_index()
    .sort_values("selected_occupation")
)

display(education_coverage)

,selected_occupation,onet_occupations,education_records
0,Administrative Assistant,1,12
1,Bookkeeper,1,12
2,Continuing Care Assistant,2,24
3,Delivery Driver,1,12
4,"Driver, Truck",1,12
5,Food Service Supervisor,1,12
6,Information Technology (IT) Analyst,4,48
7,Inside Sales Representative,1,12
8,Licensed Practical Nurse (L.P.N.),1,12
9,Office Administrator,2,24


In [111]:
# Cell — Inspect O*NET education records for Software Developer

display(
    education_profile[
        education_profile["selected_occupation"] == "Software Developer"
    ][
        [
            "selected_occupation",
            "onet_title",
            "Element Name",
            "Scale Name",
            "Category",
            "Data Value"
        ]
    ]
)

,selected_occupation,onet_title,Element Name,Scale Name,Category,Data Value
51,Software Developer,Software Developers,Required Level of Education,Required Level Of Education (Categories 1-12),1.0,0.00
52,Software Developer,Software Developers,Required Level of Education,Required Level Of Education (Categories 1-12),2.0,3.13
53,Software Developer,Software Developers,Required Level of Education,Required Level Of Education (Categories 1-12),3.0,1.67
54,Software Developer,Software Developers,Required Level of Education,Required Level Of Education (Categories 1-12),4.0,0.00
55,Software Developer,Software Developers,Required Level of Education,Required Level Of Education (Categories 1-12),5.0,5.07
56,Software Developer,Software Developers,Required Level of Education,Required Level Of Education (Categories 1-12),6.0,84.80
57,Software Developer,Software Developers,Required Level of Education,Required Level Of Education (Categories 1-12),7.0,0.81
58,Software Developer,Software Developers,Required Level of Education,Required Level Of Education (Categories 1-12),8.0,4.52
59,Software Developer,Software Developers,Required Level of Education,Required Level Of Education (Categories 1-12),9.0,0.00
60,Software Developer,Software Developers,Required Level of Education,Required Level Of Education (Categories 1-12),10.0,0.00


In [112]:
# Cell — Save O*NET education profile

education_output_path = os.path.join(
    TABLES_PATH,
    "onet_education_profile.csv"
)

education_profile.to_csv(
    education_output_path,
    index=False
)

print("Saved:")
print(education_output_path)

Saved:
C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_education_profile.csv


In [113]:
# Cell — Build validated O*NET Job Zone profile

job_zone_profile = job_zones.merge(
    onet_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title"
        ]
    ].drop_duplicates(),
    on="onet_soc_code",
    how="inner"
)

job_zone_profile = job_zone_profile[
    [
        "selected_occupation",
        "onet_soc_code",
        "onet_title",
        "Job Zone"
    ]
].drop_duplicates()

display(job_zone_profile)

,selected_occupation,onet_soc_code,onet_title,Job Zone
0,Restaurant Manager,11-9051.00,Food Service Managers,2
1,Information Technology (IT) Analyst,15-1211.00,Computer Systems Analysts,4
2,Information Technology (IT) Analyst,15-1211.01,Health Informatics Specialists,5
3,Information Technology (IT) Analyst,15-1212.00,Information Security Analysts,4
4,Software Developer,15-1252.00,Software Developers,4
5,Information Technology (IT) Analyst,15-1253.00,Software Quality Assurance Analysts and Testers,4
6,Secondary School Teacher,25-2031.00,"Secondary School Teachers, Except Special and Career/Technical Education",4
7,Licensed Practical Nurse (L.P.N.),29-2061.00,Licensed Practical and Licensed Vocational Nurses,3
8,Continuing Care Assistant,31-1131.00,Nursing Assistants,3
9,Continuing Care Assistant,31-1132.00,Orderlies,2


In [114]:
# Cell — Check O*NET Job Zone coverage by selected occupation

job_zone_coverage = (
    job_zone_profile
    .groupby("selected_occupation")
    .agg(
        onet_occupations=("onet_soc_code", "nunique"),
        job_zone_records=("Job Zone", "count"),
        job_zones=("Job Zone", lambda x: ", ".join(
            sorted(x.astype(str).unique())
        ))
    )
    .reset_index()
    .sort_values("selected_occupation")
)

display(job_zone_coverage)

,selected_occupation,onet_occupations,job_zone_records,job_zones
0,Administrative Assistant,1,1,2
1,Bookkeeper,1,1,3
2,Continuing Care Assistant,2,2,"2, 3"
3,Delivery Driver,1,1,2
4,"Driver, Truck",1,1,2
5,Food Service Supervisor,1,1,2
6,Information Technology (IT) Analyst,4,4,"4, 5"
7,Inside Sales Representative,1,1,2
8,Licensed Practical Nurse (L.P.N.),1,1,3
9,Office Administrator,2,2,3


In [115]:
# Cell — Save O*NET Job Zone profile

job_zone_output_path = os.path.join(
    TABLES_PATH,
    "onet_job_zone_profile.csv"
)

job_zone_profile.to_csv(
    job_zone_output_path,
    index=False
)

print("Saved:")
print(job_zone_output_path)

Saved:
C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_job_zone_profile.csv


In [116]:
# Cell — Inspect O*NET training data

print("Training records:", len(training))

display(
    training[
        [
            "onet_soc_code",
            "Element ID",
            "Element Name",
            "Scale ID",
            "Scale Name",
            "Category",
            "Data Value"
        ]
    ].head(20)
)

Training records: 26025


,onet_soc_code,Element ID,Element Name,Scale ID,Scale Name,Category,Data Value
0,11-1011.00,3.A.1,Related Work Experience,RW,Related Work Experience (Categories 1-11),1.0,0.00
1,11-1011.00,3.A.1,Related Work Experience,RW,Related Work Experience (Categories 1-11),2.0,0.00
2,11-1011.00,3.A.1,Related Work Experience,RW,Related Work Experience (Categories 1-11),3.0,0.00
3,11-1011.00,3.A.1,Related Work Experience,RW,Related Work Experience (Categories 1-11),4.0,0.00
4,11-1011.00,3.A.1,Related Work Experience,RW,Related Work Experience (Categories 1-11),5.0,0.00
5,11-1011.00,3.A.1,Related Work Experience,RW,Related Work Experience (Categories 1-11),6.0,0.00
6,11-1011.00,3.A.1,Related Work Experience,RW,Related Work Experience (Categories 1-11),7.0,9.69
7,11-1011.00,3.A.1,Related Work Experience,RW,Related Work Experience (Categories 1-11),8.0,5.87
8,11-1011.00,3.A.1,Related Work Experience,RW,Related Work Experience (Categories 1-11),9.0,15.09
9,11-1011.00,3.A.1,Related Work Experience,RW,Related Work Experience (Categories 1-11),10.0,1.11


In [117]:
# Cell — Inspect O*NET training elements, scales, and categories

print("Training elements:")
print(training["Element Name"].dropna().unique())

print("\nTraining scale IDs:")
print(training["Scale ID"].dropna().unique())

print("\nTraining categories:")
print(
    training["Category"]
    .dropna()
    .unique()
)

Training elements:
['Related Work Experience' 'On-Site or In-Plant Training'
 'On-the-Job Training' 'Job-related Apprenticeship']

Training scale IDs:
['RW' 'PT' 'OJ' 'IM']

Training categories:
[ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11.]


In [118]:
# Cell — Attach selected occupation labels to training

training_profile = training.merge(
    onet_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title"
        ]
    ].drop_duplicates(),
    on="onet_soc_code",
    how="inner"
)

display(
    training_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title",
            "Element Name",
            "Scale Name",
            "Category",
            "Data Value"
        ]
    ].head(30)
)

,selected_occupation,onet_soc_code,onet_title,Element Name,Scale Name,Category,Data Value
0,Restaurant Manager,11-9051.00,Food Service Managers,Related Work Experience,Related Work Experience (Categories 1-11),1.0,15.46
1,Restaurant Manager,11-9051.00,Food Service Managers,Related Work Experience,Related Work Experience (Categories 1-11),2.0,13.18
2,Restaurant Manager,11-9051.00,Food Service Managers,Related Work Experience,Related Work Experience (Categories 1-11),3.0,1.09
3,Restaurant Manager,11-9051.00,Food Service Managers,Related Work Experience,Related Work Experience (Categories 1-11),4.0,13.67
4,Restaurant Manager,11-9051.00,Food Service Managers,Related Work Experience,Related Work Experience (Categories 1-11),5.0,0.79
5,Restaurant Manager,11-9051.00,Food Service Managers,Related Work Experience,Related Work Experience (Categories 1-11),6.0,19.81
6,Restaurant Manager,11-9051.00,Food Service Managers,Related Work Experience,Related Work Experience (Categories 1-11),7.0,18.31
7,Restaurant Manager,11-9051.00,Food Service Managers,Related Work Experience,Related Work Experience (Categories 1-11),8.0,10.73
8,Restaurant Manager,11-9051.00,Food Service Managers,Related Work Experience,Related Work Experience (Categories 1-11),9.0,0.00
9,Restaurant Manager,11-9051.00,Food Service Managers,Related Work Experience,Related Work Experience (Categories 1-11),10.0,0.00


In [119]:
# Cell — Training coverage by occupation

training_coverage = (
    training_profile
    .groupby("selected_occupation")
    .agg(
        onet_occupations=("onet_soc_code", "nunique"),
        training_records=("Category", "count")
    )
    .reset_index()
    .sort_values("selected_occupation")
)

display(training_coverage)

,selected_occupation,onet_occupations,training_records
0,Administrative Assistant,1,29
1,Bookkeeper,1,29
2,Continuing Care Assistant,2,58
3,Delivery Driver,1,29
4,"Driver, Truck",1,29
5,Food Service Supervisor,1,29
6,Information Technology (IT) Analyst,4,116
7,Inside Sales Representative,1,29
8,Licensed Practical Nurse (L.P.N.),1,29
9,Office Administrator,2,58


In [120]:
# Cell — Save O*NET training profile

training_output_path = os.path.join(
    TABLES_PATH,
    "onet_training_profile.csv"
)

training_profile.to_csv(
    training_output_path,
    index=False
)

print("Saved:")
print(training_output_path)

Saved:
C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_training_profile.csv


In [121]:
# Cell — Prepare O*NET education profile

education_profile = education.copy()

education_profile = education_profile.rename(
    columns={
        "Element Name": "education_element",
        "Scale Name": "scale_name",
        "Category": "education_category",
        "Data Value": "data_value"
    }
)

education_profile = education_profile[
    [
        "O*NET-SOC Code",
        "education_element",
        "scale_name",
        "education_category",
        "data_value"
    ]
].copy()

education_profile = education_profile.rename(
    columns={
        "O*NET-SOC Code": "onet_soc_code"
    }
)

print("Education profile records:", len(education_profile))

display(education_profile.head(20))

Education profile records: 11100


,onet_soc_code,education_element,scale_name,education_category,data_value
0,11-1011.00,Required Level of Education,Required Level Of Education (Categories 1-12),1.0,0.00
1,11-1011.00,Required Level of Education,Required Level Of Education (Categories 1-12),2.0,4.46
2,11-1011.00,Required Level of Education,Required Level Of Education (Categories 1-12),3.0,0.00
3,11-1011.00,Required Level of Education,Required Level Of Education (Categories 1-12),4.0,0.00
4,11-1011.00,Required Level of Education,Required Level Of Education (Categories 1-12),5.0,5.15
5,11-1011.00,Required Level of Education,Required Level Of Education (Categories 1-12),6.0,32.29
6,11-1011.00,Required Level of Education,Required Level Of Education (Categories 1-12),7.0,0.00
7,11-1011.00,Required Level of Education,Required Level Of Education (Categories 1-12),8.0,45.91
8,11-1011.00,Required Level of Education,Required Level Of Education (Categories 1-12),9.0,3.94
9,11-1011.00,Required Level of Education,Required Level Of Education (Categories 1-12),10.0,0.55


In [122]:
# Cell — Attach selected occupation labels to education

education_profile = education_profile.merge(
    onet_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title"
        ]
    ].drop_duplicates(),
    on="onet_soc_code",
    how="inner"
)

display(
    education_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title",
            "education_element",
            "education_category",
            "data_value"
        ]
    ].head(30)
)

,selected_occupation,onet_soc_code,onet_title,education_element,education_category,data_value
0,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,1.0,16.14
1,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,2.0,29.66
2,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,3.0,21.34
3,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,4.0,4.56
4,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,5.0,18.56
5,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,6.0,9.73
6,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,7.0,0.00
7,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,8.0,0.00
8,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,9.0,0.00
9,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,10.0,0.00


In [123]:
# Cell — Check O*NET education coverage by selected occupation

education_coverage = (
    education_profile
    .groupby("selected_occupation")
    .agg(
        onet_occupations=("onet_soc_code", "nunique"),
        education_records=("education_category", "count")
    )
    .reset_index()
    .sort_values("selected_occupation")
)

display(education_coverage)

,selected_occupation,onet_occupations,education_records
0,Administrative Assistant,1,12
1,Bookkeeper,1,12
2,Continuing Care Assistant,2,24
3,Delivery Driver,1,12
4,"Driver, Truck",1,12
5,Food Service Supervisor,1,12
6,Information Technology (IT) Analyst,4,48
7,Inside Sales Representative,1,12
8,Licensed Practical Nurse (L.P.N.),1,12
9,Office Administrator,2,24


In [124]:
# Cell — Inspect required education distributions

display(
    education_profile[
        education_profile["education_element"] == "Required Level of Education"
    ][
        [
            "selected_occupation",
            "onet_title",
            "education_category",
            "data_value"
        ]
    ].head(50)
)

,selected_occupation,onet_title,education_category,data_value
0,Restaurant Manager,Food Service Managers,1.0,16.14
1,Restaurant Manager,Food Service Managers,2.0,29.66
2,Restaurant Manager,Food Service Managers,3.0,21.34
3,Restaurant Manager,Food Service Managers,4.0,4.56
4,Restaurant Manager,Food Service Managers,5.0,18.56
5,Restaurant Manager,Food Service Managers,6.0,9.73
6,Restaurant Manager,Food Service Managers,7.0,0.00
7,Restaurant Manager,Food Service Managers,8.0,0.00
8,Restaurant Manager,Food Service Managers,9.0,0.00
9,Restaurant Manager,Food Service Managers,10.0,0.00


In [125]:
# Cell — Inspect O*NET education category values

print(
    education_profile["education_category"]
    .dropna()
    .unique()
)

[ 1.  2.  3.  4.  5.  6.  7.  8.  9. 10. 11. 12.]


In [126]:
# Cell — O*NET education category labels

education_labels = {
    1: "Less than high school diploma",
    2: "High school diploma",
    3: "Post-secondary certificate",
    4: "Some college, no degree",
    5: "Associate degree",
    6: "Bachelor's degree",
    7: "Post-baccalaureate certificate",
    8: "Master's degree",
    9: "Post-master's certificate",
    10: "First professional degree",
    11: "Doctoral degree",
    12: "Post-doctoral training"
}

education_profile["education_level"] = (
    pd.to_numeric(
        education_profile["education_category"],
        errors="coerce"
    )
    .map(education_labels)
)

display(
    education_profile[
        [
            "selected_occupation",
            "education_category",
            "education_level",
            "data_value"
        ]
    ].head(30)
)

,selected_occupation,education_category,education_level,data_value
0,Restaurant Manager,1.0,Less than high school diploma,16.14
1,Restaurant Manager,2.0,High school diploma,29.66
2,Restaurant Manager,3.0,Post-secondary certificate,21.34
3,Restaurant Manager,4.0,"Some college, no degree",4.56
4,Restaurant Manager,5.0,Associate degree,18.56
5,Restaurant Manager,6.0,Bachelor's degree,9.73
6,Restaurant Manager,7.0,Post-baccalaureate certificate,0.00
7,Restaurant Manager,8.0,Master's degree,0.00
8,Restaurant Manager,9.0,Post-master's certificate,0.00
9,Restaurant Manager,10.0,First professional degree,0.00


In [127]:
# Cell — Identify dominant O*NET education level

required_education = education_profile[
    education_profile["education_element"] == "Required Level of Education"
].copy()

dominant_education = (
    required_education
    .sort_values(
        [
            "selected_occupation",
            "data_value"
        ],
        ascending=[True, False]
    )
    .groupby("selected_occupation")
    .head(1)
    .reset_index(drop=True)
)

display(
    dominant_education[
        [
            "selected_occupation",
            "onet_title",
            "education_category",
            "education_level",
            "data_value"
        ]
    ]
)

,selected_occupation,onet_title,education_category,education_level,data_value
0,Administrative Assistant,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",2.0,High school diploma,49.90
1,Bookkeeper,"Bookkeeping, Accounting, and Auditing Clerks",2.0,High school diploma,40.89
2,Continuing Care Assistant,Orderlies,2.0,High school diploma,89.32
3,Delivery Driver,Light Truck Drivers,2.0,High school diploma,73.97
4,"Driver, Truck",Heavy and Tractor-Trailer Truck Drivers,2.0,High school diploma,54.34
5,Food Service Supervisor,First-Line Supervisors of Food Preparation and Serving Workers,2.0,High school diploma,69.59
6,Information Technology (IT) Analyst,Information Security Analysts,6.0,Bachelor's degree,52.64
7,Inside Sales Representative,Retail Salespersons,2.0,High school diploma,63.45
8,Licensed Practical Nurse (L.P.N.),Licensed Practical and Licensed Vocational Nurses,4.0,"Some college, no degree",38.24
9,Office Administrator,First-Line Supervisors of Office and Administrative Support Workers,6.0,Bachelor's degree,45.22


In [128]:
# Cell — Save O*NET education profile

education_output_path = os.path.join(
    TABLES_PATH,
    "onet_education_profile.csv"
)

education_profile.to_csv(
    education_output_path,
    index=False
)

print("Saved:")
print(education_output_path)

Saved:
C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_education_profile.csv


In [129]:
# Cell — Prepare O*NET Job Zone profile

job_zone_profile = job_zones.copy()

print("Original Job Zone columns:")
print(job_zone_profile.columns.tolist())

# Rename only the original O*NET columns
job_zone_profile = job_zone_profile.rename(
    columns={
        "O*NET-SOC Code": "onet_soc_code",
        "Title": "onet_title",
        "Job Zone": "job_zone"
    }
)

# Remove any duplicate column names
job_zone_profile = job_zone_profile.loc[
    :, ~job_zone_profile.columns.duplicated()
].copy()

job_zone_profile = job_zone_profile[
    [
        "onet_soc_code",
        "onet_title",
        "job_zone"
    ]
].copy()

print("\nJob Zone columns after cleaning:")
print(job_zone_profile.columns.tolist())

print("\nJob Zone records:", len(job_zone_profile))

display(job_zone_profile.head(20))

Original Job Zone columns:
['O*NET-SOC Code', 'Title', 'Job Zone', 'Date', 'Domain Source', 'onet_soc_code']

Job Zone columns after cleaning:
['onet_soc_code', 'onet_title', 'job_zone']

Job Zone records: 923


,onet_soc_code,onet_title,job_zone
0,11-1011.00,Chief Executives,5
1,11-1011.03,Chief Sustainability Officers,5
2,11-1021.00,General and Operations Managers,4
3,11-1031.00,Legislators,4
4,11-2011.00,Advertising and Promotions Managers,4
5,11-2021.00,Marketing Managers,4
6,11-2022.00,Sales Managers,4
7,11-2032.00,Public Relations Managers,4
8,11-2033.00,Fundraising Managers,4
9,11-3012.00,Administrative Services Managers,3


In [130]:
# Cell — Attach selected occupation labels to O*NET Job Zone

# Make sure the mapping table has unique O*NET codes
onet_labels = (
    onet_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title"
        ]
    ]
    .drop_duplicates()
    .copy()
)

# Make sure there are no duplicate column names
onet_labels = onet_labels.loc[
    :, ~onet_labels.columns.duplicated()
].copy()

job_zone_profile = job_zone_profile.merge(
    onet_labels,
    on="onet_soc_code",
    how="inner",
    suffixes=("", "_mapping")
)

# If the merge created a duplicate title column, keep the original
if "onet_title_mapping" in job_zone_profile.columns:
    job_zone_profile = job_zone_profile.drop(
        columns=["onet_title_mapping"]
    )

display(
    job_zone_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title",
            "job_zone"
        ]
    ].sort_values("selected_occupation")
)

,selected_occupation,onet_soc_code,onet_title,job_zone
17,Administrative Assistant,43-6014.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",2
15,Bookkeeper,43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks",3
9,Continuing Care Assistant,31-1132.00,Orderlies,2
8,Continuing Care Assistant,31-1131.00,Nursing Assistants,3
19,Delivery Driver,53-3033.00,Light Truck Drivers,2
18,"Driver, Truck",53-3032.00,Heavy and Tractor-Trailer Truck Drivers,2
10,Food Service Supervisor,35-1012.00,First-Line Supervisors of Food Preparation and Serving Workers,2
5,Information Technology (IT) Analyst,15-1253.00,Software Quality Assurance Analysts and Testers,4
3,Information Technology (IT) Analyst,15-1212.00,Information Security Analysts,4
2,Information Technology (IT) Analyst,15-1211.01,Health Informatics Specialists,5


In [131]:
# Validate O*NET Job Zone profile structure

print("Columns:")
print(job_zone_profile.columns.tolist())

print("\nDuplicate column names:")
print(
    job_zone_profile.columns[
        job_zone_profile.columns.duplicated()
    ].tolist()
)

print("\nRows:", len(job_zone_profile))
print(
    "Selected occupations:",
    job_zone_profile["selected_occupation"].nunique()
)
print(
    "O*NET occupations:",
    job_zone_profile["onet_soc_code"].nunique()
)

Columns:
['onet_soc_code', 'onet_title', 'job_zone', 'selected_occupation']

Duplicate column names:
[]

Rows: 20
Selected occupations: 15
O*NET occupations: 18


In [132]:
# Cell — Save O*NET Job Zone profile

job_zone_output_path = os.path.join(
    TABLES_PATH,
    "onet_job_zone_profile.csv"
)

job_zone_profile.to_csv(
    job_zone_output_path,
    index=False
)

print("Saved:")
print(job_zone_output_path)

Saved:
C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_job_zone_profile.csv


In [133]:
# Cell — Validate final O*NET Job Zone profile

print("Job Zone profile validation")
print("-" * 40)
print("Rows:", len(job_zone_profile))
print("Selected occupations:", job_zone_profile["selected_occupation"].nunique())
print("O*NET occupations:", job_zone_profile["onet_soc_code"].nunique())
print(
    "Duplicate column names:",
    job_zone_profile.columns[job_zone_profile.columns.duplicated()].tolist()
)

display(
    job_zone_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title",
            "job_zone"
        ]
    ].sort_values(
        ["selected_occupation", "onet_soc_code"]
    )
)

Job Zone profile validation
----------------------------------------
Rows: 20
Selected occupations: 15
O*NET occupations: 18
Duplicate column names: []


,selected_occupation,onet_soc_code,onet_title,job_zone
17,Administrative Assistant,43-6014.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",2
15,Bookkeeper,43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks",3
8,Continuing Care Assistant,31-1131.00,Nursing Assistants,3
9,Continuing Care Assistant,31-1132.00,Orderlies,2
19,Delivery Driver,53-3033.00,Light Truck Drivers,2
18,"Driver, Truck",53-3032.00,Heavy and Tractor-Trailer Truck Drivers,2
10,Food Service Supervisor,35-1012.00,First-Line Supervisors of Food Preparation and Serving Workers,2
1,Information Technology (IT) Analyst,15-1211.00,Computer Systems Analysts,4
2,Information Technology (IT) Analyst,15-1211.01,Health Informatics Specialists,5
3,Information Technology (IT) Analyst,15-1212.00,Information Security Analysts,4


In [134]:
# Cell — Validate O*NET Job Zone mappings by occupation

print("Job Zone profile validation")
print("-" * 40)
print("Rows:", len(job_zone_profile))
print("Selected occupations:", job_zone_profile["selected_occupation"].nunique())
print("O*NET occupations:", job_zone_profile["onet_soc_code"].nunique())
print(
    "Duplicate column names:",
    job_zone_profile.columns[
        job_zone_profile.columns.duplicated()
    ].tolist()
)

print("\nO*NET mappings per selected occupation:")
display(
    job_zone_profile.groupby("selected_occupation")["onet_soc_code"]
    .nunique()
    .reset_index(name="onet_occupation_count")
    .sort_values("selected_occupation")
)

display(
    job_zone_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title",
            "job_zone"
        ]
    ].sort_values(
        ["selected_occupation", "onet_soc_code"]
    )
)

Job Zone profile validation
----------------------------------------
Rows: 20
Selected occupations: 15
O*NET occupations: 18
Duplicate column names: []

O*NET mappings per selected occupation:


,selected_occupation,onet_occupation_count
0,Administrative Assistant,1
1,Bookkeeper,1
2,Continuing Care Assistant,2
3,Delivery Driver,1
4,"Driver, Truck",1
5,Food Service Supervisor,1
6,Information Technology (IT) Analyst,4
7,Inside Sales Representative,1
8,Licensed Practical Nurse (L.P.N.),1
9,Office Administrator,2


,selected_occupation,onet_soc_code,onet_title,job_zone
17,Administrative Assistant,43-6014.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",2
15,Bookkeeper,43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks",3
8,Continuing Care Assistant,31-1131.00,Nursing Assistants,3
9,Continuing Care Assistant,31-1132.00,Orderlies,2
19,Delivery Driver,53-3033.00,Light Truck Drivers,2
18,"Driver, Truck",53-3032.00,Heavy and Tractor-Trailer Truck Drivers,2
10,Food Service Supervisor,35-1012.00,First-Line Supervisors of Food Preparation and Serving Workers,2
1,Information Technology (IT) Analyst,15-1211.00,Computer Systems Analysts,4
2,Information Technology (IT) Analyst,15-1211.01,Health Informatics Specialists,5
3,Information Technology (IT) Analyst,15-1212.00,Information Security Analysts,4


In [135]:
# Cell — Save O*NET Job Zone profile

job_zone_output_path = os.path.join(
    TABLES_PATH,
    "onet_job_zone_profile.csv"
)

job_zone_profile.to_csv(
    job_zone_output_path,
    index=False
)

print("Saved:")
print(job_zone_output_path)

Saved:
C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_job_zone_profile.csv


In [136]:
# Cell — Attach selected occupation labels to Education

education_labels = (
    onet_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title"
        ]
    ]
    .drop_duplicates()
    .copy()
)

education_labels = education_labels.loc[
    :,
    ~education_labels.columns.duplicated()
].copy()

education_profile = education_profile.loc[
    :,
    ~education_profile.columns.duplicated()
].copy()

education_profile = education_profile.merge(
    education_labels,
    on="onet_soc_code",
    how="inner",
    suffixes=("", "_mapping")
)

if "onet_title_mapping" in education_profile.columns:
    education_profile = education_profile.drop(
        columns=["onet_title_mapping"]
    )

display(
    education_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title",
            "education_element",
            "scale_name",
            "education_category",
            "data_value"
        ]
    ].head(30)
)

,selected_occupation,onet_soc_code,onet_title,education_element,scale_name,education_category,data_value
0,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),1.0,16.14
1,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),2.0,29.66
2,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),3.0,21.34
3,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),4.0,4.56
4,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),5.0,18.56
5,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),6.0,9.73
6,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),7.0,0.00
7,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),8.0,0.00
8,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),9.0,0.00
9,Restaurant Manager,11-9051.00,Food Service Managers,Required Level of Education,Required Level Of Education (Categories 1-12),10.0,0.00


In [137]:
# Cell — Check O*NET education coverage by selected occupation

education_coverage = (
    education_profile
    .groupby("selected_occupation")
    .agg(
        onet_occupations=("onet_soc_code", "nunique"),
        education_records=("education_category", "count")
    )
    .reset_index()
    .sort_values("selected_occupation")
)

display(education_coverage)

,selected_occupation,onet_occupations,education_records
0,Administrative Assistant,1,12
1,Bookkeeper,1,12
2,Continuing Care Assistant,2,24
3,Delivery Driver,1,12
4,"Driver, Truck",1,12
5,Food Service Supervisor,1,12
6,Information Technology (IT) Analyst,4,48
7,Inside Sales Representative,1,24
8,Licensed Practical Nurse (L.P.N.),1,12
9,Office Administrator,2,36


In [138]:
# Cell — Save O*NET education profile

education_output_path = os.path.join(
    TABLES_PATH,
    "onet_education_profile.csv"
)

education_profile.to_csv(
    education_output_path,
    index=False
)

print("Saved:")
print(education_output_path)

Saved:
C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_education_profile.csv


In [139]:
# Cell — Validate Education profile structure

print("Education profile validation")
print("--------------------------------")

print("Rows:", len(education_profile))

print(
    "Selected occupations:",
    education_profile["selected_occupation"].nunique()
)

print(
    "Unique O*NET occupations:",
    education_profile["onet_soc_code"].nunique()
)

print(
    "Duplicate column names:",
    education_profile.columns[
        education_profile.columns.duplicated()
    ].tolist()
)

print(
    "Missing O*NET codes:",
    education_profile["onet_soc_code"].isna().sum()
)

print(
    "Missing occupation labels:",
    education_profile["selected_occupation"].isna().sum()
)

Education profile validation
--------------------------------
Rows: 300
Selected occupations: 15
Unique O*NET occupations: 18
Duplicate column names: []
Missing O*NET codes: 0
Missing occupation labels: 0


In [140]:
# Cell — Build integrated O*NET occupation profile

onet_summary = (
    onet_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title"
        ]
    ]
    .drop_duplicates()
    .copy()
)

# Skills
skill_counts = (
    skills_profile
    .groupby("onet_soc_code")
    .agg(
        skill_count=("skill", "nunique")
    )
    .reset_index()
)

# Knowledge
knowledge_counts = (
    knowledge_profile
    .groupby("onet_soc_code")
    .agg(
        knowledge_count=("knowledge_area", "nunique")
    )
    .reset_index()
)

# Abilities
ability_counts = (
    abilities_profile
    .groupby("onet_soc_code")
    .agg(
        ability_count=("ability", "nunique")
    )
    .reset_index()
)

# Tasks
task_counts = (
    tasks_profile
    .groupby("onet_soc_code")
    .agg(
        task_count=("task_id", "nunique")
    )
    .reset_index()
)

# Education
education_counts = (
    education_profile
    .groupby("onet_soc_code")
    .agg(
        education_record_count=("education_category", "count")
    )
    .reset_index()
)

# Training
training_counts = (
    training_profile
    .groupby("onet_soc_code")
    .agg(
        training_record_count=("Category", "count")
    )
    .reset_index()
)

# Job Zone
job_zone_counts = (
    job_zone_profile[
        [
            "onet_soc_code",
            "job_zone"
        ]
    ]
    .drop_duplicates("onet_soc_code")
)

# Combine all O*NET components
onet_summary = (
    onet_summary
    .merge(skill_counts, on="onet_soc_code", how="left")
    .merge(knowledge_counts, on="onet_soc_code", how="left")
    .merge(ability_counts, on="onet_soc_code", how="left")
    .merge(task_counts, on="onet_soc_code", how="left")
    .merge(education_counts, on="onet_soc_code", how="left")
    .merge(training_counts, on="onet_soc_code", how="left")
    .merge(job_zone_counts, on="onet_soc_code", how="left")
)

display(
    onet_summary.sort_values(
        ["selected_occupation", "onet_soc_code"]
    )
)

,selected_occupation,onet_soc_code,onet_title,skill_count,knowledge_count,ability_count,task_count,education_record_count,training_record_count,job_zone
5,Administrative Assistant,43-6014.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",10,33,52,31,12,29,2
6,Bookkeeper,43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks",10,33,52,28,12,29,3
17,Continuing Care Assistant,31-1131.00,Nursing Assistants,10,33,52,33,12,29,3
18,Continuing Care Assistant,31-1132.00,Orderlies,10,33,52,22,12,29,2
15,Delivery Driver,53-3033.00,Light Truck Drivers,10,33,52,13,12,29,2
14,"Driver, Truck",53-3032.00,Heavy and Tractor-Trailer Truck Drivers,10,33,52,29,12,29,2
11,Food Service Supervisor,35-1012.00,First-Line Supervisors of Food Preparation and Serving Workers,10,33,52,26,12,29,2
1,Information Technology (IT) Analyst,15-1211.00,Computer Systems Analysts,10,33,52,22,12,29,4
2,Information Technology (IT) Analyst,15-1211.01,Health Informatics Specialists,10,33,52,17,12,29,5
3,Information Technology (IT) Analyst,15-1212.00,Information Security Analysts,10,33,52,11,12,29,4


In [141]:
# Cell — Save integrated O*NET occupation profile

onet_summary_output_path = os.path.join(
    TABLES_PATH,
    "onet_integrated_occupation_profile.csv"
)

onet_summary.to_csv(
    onet_summary_output_path,
    index=False
)

print("Saved:")
print(onet_summary_output_path)

Saved:
C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_integrated_occupation_profile.csv


In [142]:
# Cell — Final O*NET integration validation

print("=" * 60)
print("FINAL O*NET INTEGRATION VALIDATION")
print("=" * 60)

print("\nRows:", len(onet_summary))

print(
    "Selected occupations:",
    onet_summary["selected_occupation"].nunique()
)

print(
    "Unique O*NET occupations:",
    onet_summary["onet_soc_code"].nunique()
)

print(
    "Duplicate O*NET codes:",
    onet_summary["onet_soc_code"].duplicated().sum()
)

print(
    "Duplicate column names:",
    onet_summary.columns[
        onet_summary.columns.duplicated()
    ].tolist()
)

print(
    "Missing O*NET codes:",
    onet_summary["onet_soc_code"].isna().sum()
)

print(
    "Missing Job Zones:",
    onet_summary["job_zone"].isna().sum()
)

FINAL O*NET INTEGRATION VALIDATION

Rows: 20
Selected occupations: 15
Unique O*NET occupations: 18
Duplicate O*NET codes: 2
Duplicate column names: []
Missing O*NET codes: 0
Missing Job Zones: 0


# CIP 2020 Education-to-Occupation Integration

In [144]:
# Cell — Load CIP 2020 data

cip_path = r"C:\Users\Admin\Capstone_Project\Data\Raw_Data\CIPCode2020.csv"

cip_data = pd.read_csv(cip_path)

print("CIP records:", len(cip_data))

print("\nCIP columns:")
print(cip_data.columns.tolist())

display(cip_data.head(20))

CIP records: 2848

CIP columns:
['CIPFamily', 'CIPCode', 'Action', 'TextChange', 'CIPTitle', 'CIPDefinition', 'CrossReferences', 'Examples']


,CIPFamily,CIPCode,Action,TextChange,CIPTitle,CIPDefinition,CrossReferences,Examples
0,"=""01""","=""01""",No substantive changes,yes,AGRICULTURAL/ANIMAL/PLANT/VETERINARY SCIENCE AND RELATED FIELDS.,"Instructional programs that focus on agriculture, animal, plant, veterinary, and related science...",NaN,NaN
1,"=""01""","=""01.00""",No substantive changes,no,"Agriculture, General.",Instructional content is defined in code 01.0000.,NaN,NaN
2,"=""01""","=""01.0000""",No substantive changes,no,"Agriculture, General.",A program that focuses on the general principles and practice of agricultural research and produ...,14.0301 - Agricultural Engineering.,NaN
3,"=""01""","=""01.01""",No substantive changes,no,Agricultural Business and Management.,Instructional content for this group of programs is defined in codes 01.0101 - 01.0199.,NaN,NaN
4,"=""01""","=""01.0101""",No substantive changes,no,"Agricultural Business and Management, General.",A general program that focuses on modern business and economic principles involved in the organ...,NaN,NaN
5,"=""01""","=""01.0102""",No substantive changes,no,Agribusiness/Agricultural Business Operations.,A program that prepares individuals to manage agricultural businesses and agriculturally related...,NaN,NaN
6,"=""01""","=""01.0103""",No substantive changes,no,Agricultural Economics.,"A program that focuses on the application of economics to the analysis of resource allocation, p...",03.0204 - Environmental/Natural Resource Economics.,Examples: - Agroeconomics
7,"=""01""","=""01.0104""",No substantive changes,no,Farm/Farm and Ranch Management.,"A program that prepares individuals to manage farms, ranches, and similar enterprises. Includes...",NaN,NaN
8,"=""01""","=""01.0105""",No substantive changes,no,Agricultural/Farm Supplies Retailing and Wholesaling.,"A program that prepares individuals to sell agricultural products and supplies, provide support...","52.1803 - Retailing and Retail Operations., 52.1901 - Auctioneering., 52.0202 - Purchasing, Proc...",NaN
9,"=""01""","=""01.0106""",No substantive changes,yes,Agricultural Business Technology/Technician.,A program that prepares individuals to perform specialized support functions related to agricult...,NaN,NaN


In [145]:
# Cell — Load CIP 2020 → SOC 2018 crosswalk

crosswalk_path = r"C:\Users\Admin\Capstone_Project\Data\Crosswalks\CIP_to_SOC\CIP2020_SOC2018_Crosswalk.xlsx"
cip_soc_crosswalk = pd.read_excel(crosswalk_path)

print("CIP-SOC crosswalk records:", len(cip_soc_crosswalk))

print("\nCrosswalk columns:")
print(cip_soc_crosswalk.columns.tolist())

display(cip_soc_crosswalk.head(20))

CIP-SOC crosswalk records: 7

Crosswalk columns:
['File Name', 'Description']


,File Name,Description
0,CIP-SOC,This file crosswalks 2020 CIP Codes to 2018 SOC Codes in ascending order by CIP Code.
1,SOC-CIP,This file crosswalks 2018 SOC Codes to 2020 CIP Codes in ascending order by SOC Code.
2,New CIP,This file contains only NEW 2020 CIP Codes. It crosswalks NEW 2020 CIP Codes to 2018 SOC Codes i...
3,New SOC,This file contains only NEW 2018 SOC Codes. It crosswalks NEW 2018 SOC Codes to 2020 CIP Codes i...
4,Added Matches,This file contains only NEW matches that were added after reviewing the 2010 CIP SOC Crosswalk.
5,Unmatched CIP Codes,This file contains CIP Codes that do not have a corresponding SOC Code.
6,Unmatched SOC Codes,This file contains SOC Codes that do not have a corresponding CIP Codes.


In [146]:
# Cell — Load the CIP 2020 → SOC 2018 crosswalk

cip_soc_crosswalk = pd.read_excel(
    crosswalk_path,
    sheet_name="CIP-SOC"
)

print("CIP-SOC crosswalk records:", len(cip_soc_crosswalk))

print("\nCrosswalk columns:")
print(cip_soc_crosswalk.columns.tolist())

display(cip_soc_crosswalk.head(20))

CIP-SOC crosswalk records: 6097

Crosswalk columns:
['CIP2020Code', 'CIP2020Title', 'SOC2018Code', 'SOC2018Title']


,CIP2020Code,CIP2020Title,SOC2018Code,SOC2018Title
0,1.0000,"Agriculture, General.",19-1011,Animal Scientists
1,1.0000,"Agriculture, General.",19-1012,Food Scientists and Technologists
2,1.0000,"Agriculture, General.",19-1013,Soil and Plant Scientists
3,1.0000,"Agriculture, General.",19-4012,Agricultural Technicians
4,1.0000,"Agriculture, General.",25-1041,"Agricultural Sciences Teachers, Postsecondary"
5,1.0101,"Agricultural Business and Management, General.",11-9013,"Farmers, Ranchers, and Other Agricultural Managers"
6,1.0101,"Agricultural Business and Management, General.",25-1041,"Agricultural Sciences Teachers, Postsecondary"
7,1.0101,"Agricultural Business and Management, General.",45-1011,"First-Line Supervisors of Farming, Fishing, and Forestry Workers"
8,1.0102,Agribusiness/Agricultural Business Operations.,11-9013,"Farmers, Ranchers, and Other Agricultural Managers"
9,1.0102,Agribusiness/Agricultural Business Operations.,25-1041,"Agricultural Sciences Teachers, Postsecondary"


In [147]:
# Cell — Clean CIP 2020 → SOC 2018 crosswalk

cip_soc_crosswalk = cip_soc_crosswalk.copy()

# Convert codes to strings
cip_soc_crosswalk["CIP2020Code"] = (
    cip_soc_crosswalk["CIP2020Code"]
    .astype(str)
    .str.strip()
)

cip_soc_crosswalk["SOC2018Code"] = (
    cip_soc_crosswalk["SOC2018Code"]
    .astype(str)
    .str.strip()
)

# Remove any Excel-style decimal artifacts from SOC codes
cip_soc_crosswalk["SOC2018Code"] = (
    cip_soc_crosswalk["SOC2018Code"]
    .str.replace(r"\.0$", "", regex=True)
)

# Create the O*NET base code used for matching
cip_soc_crosswalk["onet_soc_base"] = (
    cip_soc_crosswalk["SOC2018Code"]
)

print("CIP-SOC records:", len(cip_soc_crosswalk))

print("\nUnique CIP codes:", cip_soc_crosswalk["CIP2020Code"].nunique())
print("Unique SOC 2018 codes:", cip_soc_crosswalk["SOC2018Code"].nunique())

display(
    cip_soc_crosswalk[
        [
            "CIP2020Code",
            "CIP2020Title",
            "SOC2018Code",
            "SOC2018Title"
        ]
    ].head(20)
)

CIP-SOC records: 6097

Unique CIP codes: 2143
Unique SOC 2018 codes: 868


,CIP2020Code,CIP2020Title,SOC2018Code,SOC2018Title
0,1.0,"Agriculture, General.",19-1011,Animal Scientists
1,1.0,"Agriculture, General.",19-1012,Food Scientists and Technologists
2,1.0,"Agriculture, General.",19-1013,Soil and Plant Scientists
3,1.0,"Agriculture, General.",19-4012,Agricultural Technicians
4,1.0,"Agriculture, General.",25-1041,"Agricultural Sciences Teachers, Postsecondary"
5,1.0101,"Agricultural Business and Management, General.",11-9013,"Farmers, Ranchers, and Other Agricultural Managers"
6,1.0101,"Agricultural Business and Management, General.",25-1041,"Agricultural Sciences Teachers, Postsecondary"
7,1.0101,"Agricultural Business and Management, General.",45-1011,"First-Line Supervisors of Farming, Fishing, and Forestry Workers"
8,1.0102,Agribusiness/Agricultural Business Operations.,11-9013,"Farmers, Ranchers, and Other Agricultural Managers"
9,1.0102,Agribusiness/Agricultural Business Operations.,25-1041,"Agricultural Sciences Teachers, Postsecondary"


In [148]:
# Cell — Prepare selected O*NET occupations for CIP integration

onet_for_cip = (
    onet_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title"
        ]
    ]
    .drop_duplicates()
    .copy()
)

# Remove duplicate column names if any exist
onet_for_cip = onet_for_cip.loc[
    :,
    ~onet_for_cip.columns.duplicated()
].copy()

# Create the 6-digit SOC 2018 base code
# Example: 15-1252.00 → 15-1252
onet_for_cip["SOC2018Code"] = (
    onet_for_cip["onet_soc_code"]
    .astype(str)
    .str.strip()
    .str[:7]
)

print("Selected O*NET mappings:", len(onet_for_cip))
print(
    "Unique selected occupations:",
    onet_for_cip["selected_occupation"].nunique()
)
print(
    "Unique O*NET occupations:",
    onet_for_cip["onet_soc_code"].nunique()
)

display(
    onet_for_cip.sort_values(
        ["selected_occupation", "onet_soc_code"]
    )
)

Selected O*NET mappings: 20
Unique selected occupations: 15
Unique O*NET occupations: 18


,selected_occupation,onet_soc_code,onet_title,SOC2018Code
5,Administrative Assistant,43-6014.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",43-6014
6,Bookkeeper,43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks",43-3031
17,Continuing Care Assistant,31-1131.00,Nursing Assistants,31-1131
18,Continuing Care Assistant,31-1132.00,Orderlies,31-1132
15,Delivery Driver,53-3033.00,Light Truck Drivers,53-3033
14,"Driver, Truck",53-3032.00,Heavy and Tractor-Trailer Truck Drivers,53-3032
11,Food Service Supervisor,35-1012.00,First-Line Supervisors of Food Preparation and Serving Workers,35-1012
1,Information Technology (IT) Analyst,15-1211.00,Computer Systems Analysts,15-1211
2,Information Technology (IT) Analyst,15-1211.01,Health Informatics Specialists,15-1211
3,Information Technology (IT) Analyst,15-1212.00,Information Security Analysts,15-1212


In [149]:
# Cell — Map CIP programs to selected O*NET occupations

cip_onet_profile = cip_soc_crosswalk.merge(
    onet_for_cip,
    on="SOC2018Code",
    how="inner"
)

print("CIP-O*NET mapping records:", len(cip_onet_profile))

print(
    "Selected occupations:",
    cip_onet_profile["selected_occupation"].nunique()
)

print(
    "O*NET occupations:",
    cip_onet_profile["onet_soc_code"].nunique()
)

print(
    "Unique CIP programs:",
    cip_onet_profile["CIP2020Code"].nunique()
)

display(
    cip_onet_profile[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title",
            "CIP2020Code",
            "CIP2020Title",
            "SOC2018Code",
            "SOC2018Title"
        ]
    ].head(30)
)

CIP-O*NET mapping records: 200
Selected occupations: 15
O*NET occupations: 18
Unique CIP programs: 156


,selected_occupation,onet_soc_code,onet_title,CIP2020Code,CIP2020Title,SOC2018Code,SOC2018Title
0,Office Administrator,43-1011.00,First-Line Supervisors of Office and Administrative Support Workers,1.0106,Agricultural Business Technology/Technician.,43-1011,First-Line Supervisors of Office and Administrative Support Workers
1,Office Manager,43-1011.00,First-Line Supervisors of Office and Administrative Support Workers,1.0106,Agricultural Business Technology/Technician.,43-1011,First-Line Supervisors of Office and Administrative Support Workers
2,Office Administrator,43-1011.00,First-Line Supervisors of Office and Administrative Support Workers,1.8201,"Veterinary Administrative Services, General.",43-1011,First-Line Supervisors of Office and Administrative Support Workers
3,Office Manager,43-1011.00,First-Line Supervisors of Office and Administrative Support Workers,1.8201,"Veterinary Administrative Services, General.",43-1011,First-Line Supervisors of Office and Administrative Support Workers
4,Office Administrator,43-1011.00,First-Line Supervisors of Office and Administrative Support Workers,1.8202,Veterinary Office Management/Administration.,43-1011,First-Line Supervisors of Office and Administrative Support Workers
5,Office Manager,43-1011.00,First-Line Supervisors of Office and Administrative Support Workers,1.8202,Veterinary Office Management/Administration.,43-1011,First-Line Supervisors of Office and Administrative Support Workers
6,Office Administrator,43-1011.00,First-Line Supervisors of Office and Administrative Support Workers,1.8203,Veterinary Reception/Receptionist.,43-1011,First-Line Supervisors of Office and Administrative Support Workers
7,Office Manager,43-1011.00,First-Line Supervisors of Office and Administrative Support Workers,1.8203,Veterinary Reception/Receptionist.,43-1011,First-Line Supervisors of Office and Administrative Support Workers
8,Office Administrator,43-1011.00,First-Line Supervisors of Office and Administrative Support Workers,1.8204,Veterinary Administrative/Executive Assistant and Veterinary Secretary.,43-1011,First-Line Supervisors of Office and Administrative Support Workers
9,Office Manager,43-1011.00,First-Line Supervisors of Office and Administrative Support Workers,1.8204,Veterinary Administrative/Executive Assistant and Veterinary Secretary.,43-1011,First-Line Supervisors of Office and Administrative Support Workers


In [150]:
# Cell — CIP coverage by selected occupation

cip_coverage = (
    cip_onet_profile
    .groupby("selected_occupation")
    .agg(
        onet_occupations=("onet_soc_code", "nunique"),
        cip_programs=("CIP2020Code", "nunique"),
        cip_soc_mappings=("SOC2018Code", "count")
    )
    .reset_index()
    .sort_values("selected_occupation")
)

display(cip_coverage)

,selected_occupation,onet_occupations,cip_programs,cip_soc_mappings
0,Administrative Assistant,1,2,2
1,Bookkeeper,1,1,1
2,Continuing Care Assistant,2,4,4
3,Delivery Driver,1,1,1
4,"Driver, Truck",1,1,1
5,Food Service Supervisor,1,5,5
6,Information Technology (IT) Analyst,4,22,32
7,Inside Sales Representative,1,1,1
8,Licensed Practical Nurse (L.P.N.),1,2,2
9,Office Administrator,2,13,15


In [151]:
# Cell — Identify selected occupations without CIP mappings

all_selected_occupations = (
    onet_for_cip[
        "selected_occupation"
    ]
    .drop_duplicates()
    .sort_values()
)

mapped_occupations = (
    cip_onet_profile[
        "selected_occupation"
    ]
    .drop_duplicates()
)

missing_cip_occupations = (
    all_selected_occupations[
        ~all_selected_occupations.isin(mapped_occupations)
    ]
)

print("Selected occupations:", len(all_selected_occupations))
print("Occupations with CIP mappings:", len(mapped_occupations))
print("Occupations without CIP mappings:", len(missing_cip_occupations))

display(
    missing_cip_occupations.to_frame(
        name="selected_occupation"
    )
)

Selected occupations: 15
Occupations with CIP mappings: 15
Occupations without CIP mappings: 0


,selected_occupation


In [152]:
# Cell — Identify O*NET codes shared by multiple selected occupations

onet_shared_mappings = (
    onet_for_cip
    .groupby("onet_soc_code")
    .agg(
        selected_occupations=(
            "selected_occupation",
            lambda x: sorted(x.unique())
        ),
        occupation_count=(
            "selected_occupation",
            "nunique"
        )
    )
    .reset_index()
)

onet_shared_mappings = onet_shared_mappings[
    onet_shared_mappings["occupation_count"] > 1
]

display(onet_shared_mappings)

,onet_soc_code,selected_occupations,occupation_count
11,41-2031.00,"[Inside Sales Representative, Retail Sales Associate]",2
12,43-1011.00,"[Office Administrator, Office Manager]",2


In [153]:
# Cell — Build final integrated O*NET + CIP profile

# Base O*NET occupation profile

integrated_profile = (
    onet_for_cip[
        [
            "selected_occupation",
            "onet_soc_code",
            "onet_title"
        ]
    ]
    .drop_duplicates()
    .copy()
)

# O*NET Skills counts

skill_summary = (
    skills_profile
    .groupby("onet_soc_code")
    .agg(
        skill_count=("skill", "nunique")
    )
    .reset_index()
)

# O*NET Knowledge counts

knowledge_summary = (
    knowledge_profile
    .groupby("onet_soc_code")
    .agg(
        knowledge_count=("knowledge_area", "nunique")
    )
    .reset_index()
)

# O*NET Abilities counts

ability_summary = (
    abilities_profile
    .groupby("onet_soc_code")
    .agg(
        ability_count=("ability", "nunique")
    )
    .reset_index()
)

# O*NET Task counts

task_summary = (
    tasks_profile
    .groupby("onet_soc_code")
    .agg(
        task_count=("task_id", "nunique")
    )
    .reset_index()
)

# Education counts

education_summary = (
    education_profile
    .groupby("onet_soc_code")
    .agg(
        education_record_count=("education_category", "count")
    )
    .reset_index()
)

# Training counts

training_summary = (
    training_profile
    .groupby("onet_soc_code")
    .agg(
        training_record_count=("Category", "count")
    )
    .reset_index()
)

# Job Zone

job_zone_summary = (
    job_zone_profile[
        [
            "onet_soc_code",
            "job_zone"
        ]
    ]
    .drop_duplicates("onet_soc_code")
)

# CIP counts

cip_summary = (
    cip_onet_profile
    .groupby("onet_soc_code")
    .agg(
        cip_program_count=("CIP2020Code", "nunique"),
        cip_soc_mapping_count=("SOC2018Code", "count")
    )
    .reset_index()
)

# Combine all O*NET and CIP components

integrated_profile = (
    integrated_profile
    .merge(skill_summary, on="onet_soc_code", how="left")
    .merge(knowledge_summary, on="onet_soc_code", how="left")
    .merge(ability_summary, on="onet_soc_code", how="left")
    .merge(task_summary, on="onet_soc_code", how="left")
    .merge(education_summary, on="onet_soc_code", how="left")
    .merge(training_summary, on="onet_soc_code", how="left")
    .merge(job_zone_summary, on="onet_soc_code", how="left")
    .merge(cip_summary, on="onet_soc_code", how="left")
)

# Remove duplicate column names if any

integrated_profile = integrated_profile.loc[
    :,
    ~integrated_profile.columns.duplicated()
].copy()

# Sort final profile

integrated_profile = (
    integrated_profile
    .sort_values(
        [
            "selected_occupation",
            "onet_soc_code"
        ]
    )
    .reset_index(drop=True)
)

display(integrated_profile)

,selected_occupation,onet_soc_code,onet_title,skill_count,knowledge_count,ability_count,task_count,education_record_count,training_record_count,job_zone,cip_program_count,cip_soc_mapping_count
0,Administrative Assistant,43-6014.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",10,33,52,31,12,29,2,2,2
1,Bookkeeper,43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks",10,33,52,28,12,29,3,1,1
2,Continuing Care Assistant,31-1131.00,Nursing Assistants,10,33,52,33,12,29,3,3,3
3,Continuing Care Assistant,31-1132.00,Orderlies,10,33,52,22,12,29,2,1,1
4,Delivery Driver,53-3033.00,Light Truck Drivers,10,33,52,13,12,29,2,1,1
5,"Driver, Truck",53-3032.00,Heavy and Tractor-Trailer Truck Drivers,10,33,52,29,12,29,2,1,1
6,Food Service Supervisor,35-1012.00,First-Line Supervisors of Food Preparation and Serving Workers,10,33,52,26,12,29,2,5,5
7,Information Technology (IT) Analyst,15-1211.00,Computer Systems Analysts,10,33,52,22,12,29,4,4,4
8,Information Technology (IT) Analyst,15-1211.01,Health Informatics Specialists,10,33,52,17,12,29,5,4,4
9,Information Technology (IT) Analyst,15-1212.00,Information Security Analysts,10,33,52,11,12,29,4,9,9


In [154]:
# Cell — Final O*NET + CIP integration validation

print("FINAL O*NET + CIP INTEGRATION VALIDATION")
print("----------------------------------------")

print(
    "Rows:",
    len(integrated_profile)
)

print(
    "Selected occupations:",
    integrated_profile[
        "selected_occupation"
    ].nunique()
)

print(
    "Unique O*NET occupations:",
    integrated_profile[
        "onet_soc_code"
    ].nunique()
)

print(
    "Duplicate O*NET codes:",
    integrated_profile[
        "onet_soc_code"
    ].duplicated()
    .sum()
)

print(
    "Duplicate column names:",
    integrated_profile.columns[
        integrated_profile.columns.duplicated()
    ].tolist()
)

print(
    "Missing O*NET codes:",
    integrated_profile[
        "onet_soc_code"
    ].isna().sum()
)

print(
    "Missing Job Zones:",
    integrated_profile[
        "job_zone"
    ].isna().sum()
)

print(
    "Missing CIP program counts:",
    integrated_profile[
        "cip_program_count"
    ].isna().sum()
)

print(
    "Missing education counts:",
    integrated_profile[
        "education_record_count"
    ].isna().sum()
)

print(
    "Missing training counts:",
    integrated_profile[
        "training_record_count"
    ].isna().sum()
)

display(integrated_profile)

FINAL O*NET + CIP INTEGRATION VALIDATION
----------------------------------------
Rows: 20
Selected occupations: 15
Unique O*NET occupations: 18
Duplicate O*NET codes: 2
Duplicate column names: []
Missing O*NET codes: 0
Missing Job Zones: 0
Missing CIP program counts: 0
Missing education counts: 0
Missing training counts: 0


,selected_occupation,onet_soc_code,onet_title,skill_count,knowledge_count,ability_count,task_count,education_record_count,training_record_count,job_zone,cip_program_count,cip_soc_mapping_count
0,Administrative Assistant,43-6014.00,"Secretaries and Administrative Assistants, Except Legal, Medical, and Executive",10,33,52,31,12,29,2,2,2
1,Bookkeeper,43-3031.00,"Bookkeeping, Accounting, and Auditing Clerks",10,33,52,28,12,29,3,1,1
2,Continuing Care Assistant,31-1131.00,Nursing Assistants,10,33,52,33,12,29,3,3,3
3,Continuing Care Assistant,31-1132.00,Orderlies,10,33,52,22,12,29,2,1,1
4,Delivery Driver,53-3033.00,Light Truck Drivers,10,33,52,13,12,29,2,1,1
5,"Driver, Truck",53-3032.00,Heavy and Tractor-Trailer Truck Drivers,10,33,52,29,12,29,2,1,1
6,Food Service Supervisor,35-1012.00,First-Line Supervisors of Food Preparation and Serving Workers,10,33,52,26,12,29,2,5,5
7,Information Technology (IT) Analyst,15-1211.00,Computer Systems Analysts,10,33,52,22,12,29,4,4,4
8,Information Technology (IT) Analyst,15-1211.01,Health Informatics Specialists,10,33,52,17,12,29,5,4,4
9,Information Technology (IT) Analyst,15-1212.00,Information Security Analysts,10,33,52,11,12,29,4,9,9


In [155]:
# Cell — Save final integrated O*NET + CIP profile

integrated_output_path = os.path.join(
    TABLES_PATH,
    "onet_cip_integrated_occupation_profile.csv"
)

integrated_profile.to_csv(
    integrated_output_path,
    index=False
)

print("Saved:")
print(integrated_output_path)

Saved:
C:\Users\Admin\Capstone_Project\Outputs\Tables\onet_cip_integrated_occupation_profile.csv
